# 独立放大版 RMP 语言模型（`main_fine` ~242M / 61M 激活，Colab 一键单文件版）

- **架构（对齐 E129）**：100% 零 Self-Attention `WaveNet`（12 波 × 16 模块 = **192 模块**，每波 16 选 2 = **12.5% 高竞争激活** + Gated DeltaNet 快记忆 + 多尺度 `wave_ctx` 路由）
- **参数**：~242M 总参数 / ~61M 激活参数（GPT-2 BPE 50304 词表）
- **数据**：英文 `FineWeb-Edu (sample-10BT)` 流式加载，不占硬盘
- **精确防断连**：每 250 步原子存盘到 Google Drive（`MyDrive/rmp_main_fine/`），精确记录 `docs_seen`，断连重跑 Cell 5 自动无缝续训、零重复喂数据
- **免上传依赖文件**：全部最新代码已内嵌在 Cell 3，选 **A100（40GB）** 或 **L4（24GB）** 依次运行即可！


In [ ]:
# 1. 检查显卡 & 挂载 Google Drive
!nvidia-smi
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# 2. 安装依赖（Linux A100/L4 上安装 fla 融合内核）
!pip install -q tiktoken datasets einops flash-linear-attention
import torch, fla
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| BF16:", torch.cuda.is_bf16_supported(), "| fla OK")


In [ ]:
# 3. 自动释放完整代码到 /content/rmp（无需手动上传其他文件）
import base64, zlib
from pathlib import Path
BUNDLE = {"dspn_text.py": "...", "pool.py": "eNrdfetzU0eW+PdU5X/ozdQu9xpZSJDJTkg0NVswcbYqeKmQVD64vKpr6drSWpaEdIUM85st8/DbxgwBTLB5JoBDwDYTHsY2UJX9TzK+V9Kn/Au/c0737dv3IT8yu1Vby0xAurf79Onz6nNOn2699957zZcrjStPnaW79s155+mdXzYnmstvmst3G48fs4OHWOMHeDTjPJ9urlxtXFmyR75pjUz+sjn5t5Fz777z7jufJ6HDUClrpvZVLcMy98GrXzZvdLOttUeNi6vNuzPs6Inj3YzDh7bO6L3WyI3W2a9Zhu1n9tsfmsur9tsxdgK/PR1pnV9ixz47TlBmnJU5Z3rWmZm0F1ftmyMwMBtEwM7CM+faKjR49x1n8RE85tAbNy6wzN9Gzp5g9sXbzp2XW29vOjNncczJH5y5udb4TGt+iRksxZLQyh77xh69zxJ8Ls03X9vj6871lea9HyRAloN2ZxjAtafuODfP25dnaNqfJ+SkhwplmPKMvfKqMfqQz31rbQQnv3THXhnDrm+AECvOnTXf5JWpy4m2zr9uLM/zqdtzj5ypKY8gzJm950xcYt2MUwEAb619bS+/EHzgXITRndv3myv3WAVmWSxVhrSv0hX200tWGarS15quA3Sc4fT39sSYc3WVZUpVrRJjRjqvA1p96Tyz567Zk7Ns8JfNBaDwy5f2pQmYfWt8llVL/daQMaxBH3aAWeZQWYfWj+zFJc4Ze2zUfn0ZKGDPXINxttZm7Us/2GfnGq+XYfpHTbN8wjQHmTN/p/lm1J763pm9bX/31L453rx7F5DnLQFgY/ku4NhcPetKGpcARBKaNX64tbU+aT95sPWK5FVhGRKcmtqX/mKPvuDUwIc3x+2RzcbDDcDNeXKf4+Y8PWevbBLBF/jQOH0k4sxk89nt5uvXQInG4wf23Mvm6nnCY2t9pjkyKqZcLhhVK59JfVGpmTD7K83lJ42N8xyqM//CFdMFaNtYf8vx3Vq72Fy5sLW5Cg3ssY3Gxtcwg9bYZeXrDFe9+CHWnDonCXDxtn1runX9hbP83B5dArHgE+ZCY4/Nbr267lz8vnGHBP7Jt427y87Iw63N6/boBJ9tY/EstpxaAIzECP+sjvDee++9+05+qFyqWGzIsHLyi1WqZHIx/k+8WGRGlRWL3vd4f62YsfKlolHAV5+8+05/pTQkXtesfKEaz+TMzGC5lC9aTMD0nojm2Wq5mLbMYdniM+O0WUHRfvedrNnPzCEjXc0YRW04xswEiGuMVcyqaaW6S0Uzxo6kPnhfP/zuOwz+wETstQcgrsBQYL0zMw5i7mz8xZ5asl/9CJLAzLQF+jGQtn56aab/ZHUm/wySryV/nrhs6D+9HE5bQLEBamMANbc2bzRXbjbfvGpcnbG/u+CaDBASdoSB3oIekW27AX+DAoCVbN5dtcfnGre/b11/jsoMbJp/trV5h5lEaUTSYB+nWAI04UVj4RkYBeTa/B374pg991feHQUBJOji/ebbK/bCLfv116CUaKfAag1zFIYPM60vxr6IsaOg1gtmgn8/qqNFo17eexDQi84i6MVFVIHpWbSWiA6xHTHK9wuk4glBSPwz3A9UGI73F0qGpene84pp1SpFeB2D/3oOx1hnspe/FQNSt2rOKJsS/BcsBbaXGcUs5x3LVxmyTxnORJqzDmCxOyRxJp5gnczQ4cUwjpXobYuQie8RaC/ICX9bGACgXBz7a4WCRiTRYyTk8UJpQDPgS9Y8lc+YqeE4/6BLpCWmxZIVxJYgFwbiQ0Z10Mym+/MAndoDOQ4BHXW285/fMHO4rIHoHYLmP09OoHjdkMJmvwWtnnSefAe8J5s/tbU27Vwb5ziUalY1RjTrwekm/FTpL1WYlWD5IqsYxQFTSxBjjugqdzPIJiSZlTgMbfezI5K0H7HPaHaBt5naULU2pCXhPXXOcCb3uOzHP1YlL0kOFKtqmRiD/2et02UzxR/3lUqFMNXj0LOgcvWYhINU0j5DZFwGg0h85nIbnvbqPj78J4CKMT6XfZ35Yv8+XVcAnwbAWtKVKjFEvohT29eX/49Y339kO3/fl8/ui7FjIOIZlEMPkc/0nng87uLRoUidX5RPC834iFgVN8pls5jVTgs8hMhysBnAk/MzqZPo4v8ysMJUWfqPw2WzYh0pVUyNzKJi6T5PoC9A7sUNJFYSv3LPy/W5ZqLcCFisuQOB7gdYsqpZ6I/XwQoJg9GNfgL3rrbW1snmcSdINRlomNPpfDFvpdMaQoB+MfYl8BW0K8Zy+WzWhJXiZIz1mZaRBs9BFb5qDWal6XEJIbl935QE4sHImgWO+lfQmT70uR++Oig+FEoVI22oX/oUJHhndJeK8eNGxRgyLcCK8wT0JlvUYEpZ4L5ESgfXh2xH9WTF0uiVKlkCi2iIZ8wKuFvdElSw31cHd9UPMAp1rUNPFEGUG5c54J4KvoD6VVTi54tlqVooej3YoBeFVRc6llBkGUaF1p/EB8xCTQsoi5WPFfM5UBarmANtAcAuA5KoNIIcCrY5AJWRg3h6JQEWY/Afqp+F6scnF7dKGmARJxuix8J9crFijvoUsRM0lWKgB8bOqUYktIDkwrCHsrGh7EmEPXQSYQvIx3VVSY+DQSPl1IB9x0rZWsFU1FSs2zW5YH+po2exNm0/FbGHDAN4/AD+awt9APRNeQOWk52TyH+MTdQnJ/EJOJyNqRfOyNmd1XQwWttOVlKHQHPUmE5RQViJY+T3pxJxGBMfG/AxIT73pZJm56FtlTwott2CmoPiXxxYfMRxxEcaSPnch8JOc+DteVPRihoEhsmA8YQ+nEPduzU06ADgAOi4CGIws1A1fSZ5J5MXVPHKdsbmS+SA38J8GVL1ijmQr0LPdF+tvx8A7DOy2Qqw6ZM4BnlGIX/GDFowAAo45odSoOA7guvLG9V9rh4I07NzL1hqs2nw02VP7m11x2Bk9OAOsDAQEUIBRSiI8pyi1r0LPJxCv5oiOvBfeZDWujEnIq2oAAs86ubyG/aJAWwKhVaTqnEkgSSmCqXocx0SVYCFWZBCFFe6eR1UuBVYxl2QNZBRiHopSvFBJXPqceuTOEToaR6ix5hWE04VuA4xXWd/cCVHslDSSeg/sNcDjtF5CoYQ3VA44pkCYAB6aOF/Ja0iDOkOrqniDCxQVmDFWXwMHEkbWaNsYZ4FeONcecU5RHmLWYhzmg/OQhTkzD+EmFtZcrLD6HQBdopvz5cHkDdEqzyoubYA5giUzgJtqz6RkYsWiWW6kB80EaLfuSTwhw7qcYgYLRRRrRMMFIxPgqhQCtQb2dPGvfeGxL87qKni93kN6xIrN0Him+UBz5z5fVQCDhEXTtf8UKdZwzj4GKnfvHu/9frS1utZ5+oz8OfIDwsJZoWsYozVVREE179uVLKeEJK8+kPmSLkUixPMp6YGbzSeOxCnBxgAknMh4X6i8vlWDNCV4gAFfJw8+Wp6oIJWomj0Fcws0AbfqZYgTP7+OJc2lFkwqZkcSLCgLGEUMXLAZAdgwvp9JsaKVXceqNVaTXUK0McHxIT0u8vUl7qfjnx04hR40F4OEsmBSSQ1NUlrMqbD5q77kamjR8K9G6GSkRKmMFyDLh0sB8IN3slBTn20CmhAAUP++ky718VqxMJIvmO9vc3zUydIBcF+3v0PIi1UIka7qz15IJyLXCS5MKmSlylanCGEOAarsFCxvf8BowUrjgd3yDSK6YoXRvq8THDqiif3CcGmZRfQAFtpDJW1oXwxxZ3hAC9q5SzZMUD49ywhhTGipWJ+S+XTaU21+PKdawTJeQHWIfgOpgm8Oz0gkQs4LrroTwMyHHmcne6bQjA0AWOL8LgrKLwpNwSu5geKGl+sudgDAjhEaOF2F/v4UK2Q1hLxDz/UOVB0BgEcdVJNEspMlYZUuY6uMWVaG2PPYcxCaYB1w6qN0ZO9PGOPLrm54UlMXz3ZEPsUK1e23kxj6vftX5zF2xwC+5g1FpadO+Oog8kDSZGmk14w/im7Ou+i785TTofIpwTxJJWcw51aGSZWVmlrdiZhlcEcks4VTnpu+IxTUPeFq8QtDg/AfczAjZa0lqIUR+Zreji7BSYXzBIgFWOHQBz4VwSK31VqZ0pD5XzBTJeNSgTVPcX3t/MFNF8Zp0wMaiLjmc8Pyp0O2jSaoYwC54596YL99BZmGX+8Bzzi7iGrAzhol4jHu36euJwUOzxgN3+8t/XmpvP4Lu19LMBXWGGaKxvO8rT9ctl5fpYNpzEv0VeyrIJZNDODPN+OXt3cI2hIOTEcy42jJmWStfnw2+bIA1g3t9YmneXncnuEkvBP7Hvnmy9vN9ceAfQ0opc+WTMrpwnAWYBnj43ybRVozndMsCGtemk+G2o4+RDWZnhjL6zbyzdwUwVQGX9kTy01n6/aE+ONyfHG4lnWxeyRWef6G/vJdTcNPwNtYFloPNzALtB4dAliPjeL703DefnUXn/QuHK7dfUtDDRQKdXT5ilANXUcLRDfdoLgkR1HJgjfmDY5BEPmVmAMe2K1sbDWOoeDAWh0Y1P7+guGtY+hnoEL/fgxZqSRaTP7quUKiBW8ujmO+e2JVZgA77hzWNkVY0dDAREPK/cWSMb8C6BIXEi8Y0whRQITupUB0EqAgqkfEGkTn1ZAclNuhEQdakUrX8BXGWs4nTUzBvSOf/jPgI9RS5cr+VIFYUQNrgghBRgx5smN7wlAFm4WJZwgDBtMIVI5QCqLQ5MblBksW7zb3xUyd+0lXu7aRbDskdUNyomy4gufRMxtKuaBGypKN9lDNJbtgquRpKib6ZEEBYjqW++FB+I3grtA7n24A6eyw7USrY3rzeXv7M0R+9JKa+QSCHFj+nHjh2luAdgfj/0L6JRkGvbiCoXqUEeZt1de2cu3Wt+MgjGyR+9DBIQwYJEZZvbZGxzAgooTavnmHO73SLDEZAaKaI9u2suvIDRqTE7Yo0+31n9obF4DfbefnkNIzPl2BGxU49wr0DQpnsy5suLMnLVnrzrPl+Ed7mdNXANrAGYBo4Iu2qOemG/eXcKQlyZpf3fDXl2HBu5ukmdPvJShQA/dGUzIax75KPUh34uNGp778FrCK50aIhfI4SZGUCuacTBlAM1iioiRIooH3mRTjLdTm8i3wUXMfY7efA9fPjM6bX1kcOdDvu9FNPNV8P4so5iBmNF9AR5zIV+FBdWqlWGJ0zn2Lii3lY5Z/i5vcFggzYrFQDa1AB460gGUrH+fNyPOqdbiSOPJ5Nb6RdaV+lPXn/f54hZJwZxRze2LioJC+ZacZfTJXAsmerSDv/2AFNznc6jAByLCIZ5aGNguKXUUQfpyUkdDDmEwwRZaAeABmUFpELfP7LviR9IYifK2eTS+BgVTaT60kdF7B7wTVAJRGoDQNxoK7Yd1R4U2v2HgOfEVm7sy9r1bjYUpSoBRfusSOCNLuNtzbZU329q4CCt28/lL3D2+tgoeQuP+rcbGnD163h6doHDzL/a3q5T3+r+XQKwn/Mkg2qhRskCFUnGArBMuoyT+3NshBedNDL4z2o1Zly52AAbYCUeUScCvHsgiSc8BHLLk4b3Fq42Nr51bF4BhrUczuPf7YgocPmTzj/dw+Zp/Zq+tgcWH59z/kwsT+695Oe6/16nKY6r1zSWWNN9n5P9N8sqP5sML4JnCerf1ajKQfchbORaI2A+H/R7DozQEiEOlfFbzQoi+tKGHuwBifIf1gNhmTdA+qyRTRwfQMBQ6y8AVPUTz/Qi4RpI2OTsZQgZw4RY+zETg7QrEgGZ4GCV9G8LAh+/XW/N/lZVqahUO+7eD0svGxf/qN9y/RH8b66ncZDXwi7TunAwZMGstQtgJI2PlT6GR5C67z2cAHre+3+AegUSBO08+Hx9X9slZ4VZ5MRffOrLfjlJ1noLQ5EJr5AaWpKw8cX68inVvZEsArcbNu43FO87q2tarUZ4f9zkIgKx/D79b1d9cUPWUl7j41r2yg66gRIk8MKpvitV1kLwiAuFikAgkvTClx+eKmUJcb6F7lIxWcz3wpucwb93bi2IST4SBEdt2gAVzJ2Cd1PowAUvsaEKJu2Ab4MOOpi5fTXNEMfmU23l/p78/aE5hofctIsrqgTJ3icyHLH+juqet19/Ymw9Arshl3XFNgC/lwKiYcg2Z2JAnYJStGjkD5DPuwRI2l984E5e4I4sFdCTuzbev7ak7TEPJn3spcnVqVSJ8E0WPOq6VEIDfuoQk4IvmxFhrdBYUYKfZDpVOETcCa1YIR7CxmKygmB2NtNDVG1tr05iGeHwX1BOLDyHe+O6ZszYKVsVePr+18RjrMigNvO0WWOBlrWqmq4V8xgzukO1ISzUdgXtibi5C5isCSQm0CCIvgbaJV/zyulYKW34MYIaqAchJm8YjnN1g5gZmnL1bG6MM3NKcWWE8yuLlnFtrT5C6zzaaL89Le/hv2qDOOJaYDiS8MctB0RHrTv02efBAMnHw/bBwC2z3LJo7yGSdtD09UCj1Gbg1Ygl6YPHM1psF9vmx4yfyGANUWGNppHFtDr26uW/D4pjuzw+bWc5quTGRxuwcZpvUl5ovzxfdJJjvc/dFvAD9cOTWAi8fUXe1ZJjo7b1kyjXaQ8SoyZeaoLFjrEy7j1iQlgisCXlvUeB5i6DthdisgqsOgAjukuQhgkHIH7t5aTTh9R541EvLSD7CjGP7/bAMhBYBfNEpRhMAB900PDfLEdAiaeRN3C3uom8awSZaRGRxBZcFIqmUOyVyTIMlPGnyirFGlm86p7P5jLuL0mFUBoDiHR2DdfwUmUSK7B7s+N8kjbvbUa9a6s5bu/30oOxSDPsPIsUQLb5VJbl01B3jq4q7l35QLc7LBByYPi+XdlTWJQpkJA4yH6KEELJ7qFuI71oH5hnMTFCDi1Z+oFaqVXkSuo0O407J9H1fxhMzXnyh31pbx2MS63PNB2ONhWssEY93d1IpYPMtBIRPWjdv4TpECwCYJl4b3Nj8GpPR4B0QDC6WmD86+9YenWVHvjz6L6yrYpRzzH591V544yzeZl3Hv/x57PKR41+CtzHDY4zgHgsQC/fSA8sX6H/ABMlHJAUV4Ej7XfhI3VN3KX1C6SOvl0QQMgk2G2vIM2lMT7obuaT06eGToQ1xmBqE1bACyKgLicrPzAivgwq5LvK0IC9JsS9dxFaUYORrPo5XNOsR5FIKpXs61SLaryrpuifEPfVevxjyFCkJomjSRmdS0ToTlOkIizd8MibwBjxkAb5SqgJ01I7GdF2SM5ALA6QFifW29lRNOQyfxI3wYbc25Q9EA5l1cJGJFA3f1qrLze2hRUpTm6Icd6btoCgC520Q+QQOj9hksxVRhlItREkaHrtanpeShptKo0vO0zvOxDWuu9APRAgLwcmVP9CaX+KneCCw404T3z2iEhIUSvXwTt3yBXhoTdQAcyYcgrrx5IQMasHolTsHGT+BI/fwBJwDgvL7B+m0ARv0PaDTWCKxRYlpXg6xa31AgS1sYyNCQh+V7PTKo7j1toxiTmvDaJEYxai0WuiVm7MBUTYLoYGjUrjewNQWc7c9JAl1gh6EGcpMev01LOxCUeqBfsF6Lj18MIIHx+khd2gZemJ/ZL2/aQYdKxiFlx2QN+s21Nk/Me0/EVYAPKb7XPD4ucc3o4jZ/E/x6n+MSb+KQXtjjibNMc+LILH9DCMebMOtiL4iPNsL85RjHKWyKNjDg4FUo+c7YYGDi9K13RXvKXU4/vK9QKUeAOOVeqyDZhimG0Ralciiu22L7bBPZLldervFM0pSxPgc5gH+LxWAIPq+kqs2pTjuKB5nD4ekAo9eIcOixcYolHOGyzgB5oBgu/gqQppBvU1NY5qKj2Cg4OQodcsPXeEoSCWa8X6m8WEPqMMSEUNQxIJKxZm8cNAKrbbuy3SFgudK+xrG4cgaRlDOqHVUPRfHZCFJ8/nLxrlXv2wuQC/RQGf8WJ39ZJ5OQk4grxnmyye+gSBdnA3m5+94Ro0WVJbzbbg6K3Nba4/cIbO6KAxZHGlsTMjR2xXnb3Nkjuq7TKNSdLn8qyoqA0GP3+vZjaco6E696XPP4YNuEWDPQf8KzdGVKCgJh71siPiTa1sbF1sLLzjlMK32cNqPH6rbkNgVEOWfp8xsxBzIUBLWiV5RzjvUI06P+Wsbe5JtGrj7FBHbhjQq2TV/xYbrjO6O9kQ7GT+1cc4lTMGVnXzzsGOeiHDMqRgxFa4ZD6Z4lIRAZOSqdKD4j1Jv0lD50oY4Wc1Nh3ieHRdiDb+SQKkHeMBgZIpW2ihg4UBtCC2H0d+PcQHFB5iVEukn/39KpekXAOMErdT8FQX4EWhIXnjTEOUBmicLfpvutQywLPNFGnczYWD6V6pSotclsyd6YcIHtlSis2cUg6E00OEqHi9qbSTMJ126mnnymXAl/uMsV6LpbeNoPeTvCUDt/XcxBWSgm0vjfYLKRsQjZrlBcZhzbSakLkZyTkrAFh2q6dHOq5tM3n5KO4UtqnskJGM4DrNOy8SSUoIedcjhI9mVV46js05Tix4HltlfOYzoqY6iLunhPVhOH5eZSFwXU5lNjrlQlSdenb8ekTZo43rvMKpvwF2NtZ0yi/d93nl25Wi76oCht9kGWdw3KJQyhnDFehJ0rF1xo6ULHd7amb3qTM9ijSleC3C5+fBbKhi90VhfwQQdus+DuAm88IIdOf5lZ5fM1u2JmiqCWrAunwrf3Y1bo5qm4ilRyhi1jesC5DsleMhfQgcPFGBjvjxyLx9rFHFfRZCJhE927sV3wWfROhixC7QrpQz2UgUruPezreDmcCZn8C8NFgMDVwJwrNVMuPB4+TqqRWM03P6ciodNvFgbMgso30f1WDQc6QghPK+n4gAFXuj+6XonYCLLP8RyF4CBBxVg6uHmJ9q0PhHVOFen8gGLTkt45VY9SleZBqCDNciY7Y/WiMsU0mcQMsDvQEaJIzRRrJS7PbxxzmscmXjYyf4XaYFFLYCIu3MbfahRSngbGSjWkecR/Ehg0vJEgqeUFdZXgyyvbqdAXrWvtHS7iUnaaBelS0nY0/1FrUaOBCCK/5zg/1SqlptXpn+hdapaaAePaxkqGdcxVLHtNIyPqKlD6jRGcGC9/XjuqXExrBg1uj0/eAZNkBXECe8aHE0SIoZcjikcU5UtRotSxTSLQPeiWzy+J7vOsdAkGttTqM4JJFBBJ9Wn/JGqL7UDxwDH4YR8gMO1V2i/OleDavwrlHgPKtxWgSMJWXFvW/BdEeO5nvRIvSPkn8I5OlrK5x7xi4rE9pFbTcJTCiILv7iEa71bE9gan3HWRnmtScS8iLtuuOud4Bve+XzjUeV8Y70ddyN59b+AT1hSxoG6tso7OyBitmFwNahJ2NOjKDOCzbusjySJwDxs+IhqdGMRwcqaAb93hS49Rrbu651PLmb1NgPJ1WcbJ4f7ZmiNSlV+8kzg8ZEbWbuIQKtOaOoeT1MyNgtreEpi5lVrdNZef4C+KM/hqIUu23nUMj8TiJN95dqRoV47QN7VJhTvJ+WdIfjoBD3Rw6mrYNRetHy3pMjMQ0LfLvAUaWmQhEG3CxfbYPq+WvGBF/mM3QOnDtGggXWKidoNMJlGSertD2Fn8OxhlSddYgwPjn6h7y2dCEE9aKGXvKexReCPY7eptjAzbRjdIfOTCDdcdYEZIzd7vMdTynyyMskEEpTPDPrKbCh3ENwvk9stge0bjAajTibTCLs7e/yrThRr8qwyIMUxDp9a3sOZ4yqMhlP+BAuD07mS5S1tbhWNhA9g+YgRtd5cbGXtKtZ34h4zr66Q6V/n6qqyJWVUBkyZuhnEfQhApkeO36sQUITRkSlmvtzibX03LvBTw/wQcZuD1cT3PZ+jFsiKE9QhSuhe0TmWq4Z23/HOxmcrjYcr/NqTX3EAm0rkEYfI0ybcaMsz05wA7uHriRv2/de+Okq8H7C/f/urMkBo+vvbYQSvgoVmWGLM20TtjbUtV+QGCV9ryP7dXD7AG5NWUxf/hpG9eac1PmfPXENxoJWL38fIjp+2cqUiwyKLKyu/bM6oRVFzK/brq7wwX1wIitNh/8iOk6bT3Y5XbjtXJ4LbPdmSNBiiR7A0iiwFWhCl1cep4Mk5nf0/AcZ7xruqO6WnpKk11OyxrFZTSzwDdWtZ0yyblE/3NDyOO614c0tSPazBYeKxDHVDoVAo1bl11OqnetxL+yQoxRQYfVUAClNM0rRla3gikFBFR5w18wRO3ZUWo4pNXmWTjc+EOgMpBnAeKrbiyBZ/zyuS8UY67OYhynNb0kLC2F7LiHkFU2EZA3fUaWTX78ZTR79T0CgNQguQEKACYfR79QQuEYcm8g9KbS5fa/x2kP0+5RW2wrCRJQHKOmYUBxU5GaiWKpbm/9bJ8ekADKUFQyGqZsA1zBcHRCyqLnAlqgyHGcHYNMLHhErEKVTfOZ06ENTUsDcnv0La0GEDvuvGe+NnpQBkF3dPoJbT4XpAk9dIYhB2Fe+WDlw2AQ24vZSXTSTiSSya4tvC7koVUeAnVsttF8ugA1cq/jqXQrg8mE8ggeB5WrqYQl4XIS6f8C0iPaViL59S6A4KftME5Xc1hFvlp2uruBvlM75/97UTCKCeL2bBlGUAU3hB+y+fxtipUsboSx387QfKDRTu/j3t3LOfxy7z/fefJy67HXRxiNvCS3q3Xk3bly6yT+mWane339qfZD+PPGDW/k95WR1euHvtlb3wBlbOT1VGHiEOlmFR8fhI2PGRJNHk7agxhkFUgm7mS4hQ6TfsSI+FaU08wWV5mODiu3xX+jlfQAuAHbhX1a0hPoI25VO0v4e/wL/RGNEz/N7J3/Ti+dRPg5d5dJtW5F0e3k3vzuKIc21V3P9+4wI7wRrrD/nN07jIEbL28xm8yRAJLu44d8+v45kS/8Wjx/j5uEls7ExdtV8/tSdWvYs96DEoD906jzeKUMPmyoY9vu7d4UFfnelN5+J976h8jnmHnERJBg5P+uveV0raOaPeQk9FuT7onx+Cbv19qa+8Q3b2+pw9MY9VjV95tzv7rihZmcOsDD/gMzXF7x9B9fffriyqqnOssTHXWnhhT9y0H04LKkzdUQnhFqZMYA03QN540Xx7G3EdfwTAmyMzeE/k3GzrwUTAvETcwHEMr5nIpj7E6zBSyQ/c49ep5O/gyUl6Yg710V0cUqsC13JEZAm6U+//LsYGU+/HWBfeXnGUAJSzqQ/gSZkPVHZHOvg7ed2Hd8HHbq/tAF4k/jsu72Bl3Iiu7/4iD7zuJI0VM74HVQv+Gfp7L/uomuV0Pps6FL73A0NKM40cEV87OtJ7ugvkmFhE+t0DDJlcrejepVEulXCv7RiSlYxR+HizmHRM+c7n7F72QahjastrGmiFDTy4XdC2C2v7ybPR6FyNqBJrX/FDpcAP8KgVrcb8HDPdoQZKdoN1H9AG97tQQKkCswDy8bP2fxzqM7Poi2gk2yTqgXPanNqBeAK6InlFLO0GdEN98bqZH8ghcawsyadrx+3vHjZXZpt3lw7Ap8bGM8J5Whyc6AbLD26svbLZHH/G1R4vtspbVdaYGGucW1Ovl8rIOxOOEboRF4nu5dYEvqojvConCbf2n+FZrR4YLeZd66Te1VPOohpLHUb99cZxL4whFca/fAocvvQlJe9+IS2mv4UmBy/2SEXe8ZFSLufwtFp+iqnaG31DTMr7qOio+0HVU4WuQl9DN9ak5Ce913dgHJeGp+da9y44z6flgXFcR/HceDIZvEZm7oGzOClu9/3uGT+lDT3xcnfwTA7+9PL9f6/7nAKEv3YerznjVwIsTttv5uFz8+3i30bOwofGX8XPhYgKf27zgudL6WFM+fIpHrETT3sOgkf/Ph3HDx/c9tf28S5RoXg5BxYhQuDg62f5omlUNBQw7irRKGnfKKGAwL07Q3Q+Biie9O9rFtMVfpXaUb4BIw2SyPWXs/I52izxFAFhVWwX3uOWDeR/+/uiptbfV/bhwrsiMh9Js8FrmTW3vbAa2zSgWurApNH2iJFQfbuxho/PMnCXizJX2nLlFo0+kyEEmkRfH2Nked4VhvhXUHIrb53WdnEdiq+joAJHjNvX6GmKXi4tInJn2110zkUl2MuoDburEgZ0jH6rIsb+9Gff+RsS00KpWhUeUa5OhVyByC9XlzXCXT+9BGFgQW9yofl6hSfHtta/s+cmG2PPZbzHz9Ojzj+at+e+bb4YRR918bGz/BzzlniJ9yz/LZ9gNNhFQyJ6MYZSLm8Q4jqkx3yuf4zPMHiQFoKLHFD/Ux3VyMRijwoenzyTL6uwfCqv6+EjGJ9ikuCLyHMMIB3FWqB4zhpAIYiK0AL7BYUyRUuw5qXd0vxcVsvVaQuSghS8RbYDOOOd80FnoeuwVqfEUtS7XuXW187gDgXSE4mJ/+xnnRqi2gF46KIm341mQVYDJ4JFWEU9D1Bmpsg3QLY71arSMmueclNRnscgUmmBM689hfBhV24PCxRNe2t3bzgZGjZPskxSizrIygFhNWmXu1frPaIgPKseWNVj/rShcqRW0a2qOSBoYLrVkVaa3Ar4t0w/6dHfd8p/fXaV8vUa3fqRUp1Kb6dHOJk7FHAec+l8TCkHBu04k8BrbRPyak8+y0SvZkbteXeYQrCoiB2dLk/QdG8+weLoYXkYPl/UzuDNNXhepejfiKZV5JQvRb5ztbvv54XmpniiAh3isVnnxyWItWWAiaZl/iFvz+NBv0T096UxE+q7toUfsRnmm/GHk7zWOaJggf9uDV3ooaYxeIJJPUfZdlsfKTTsbj3BKqcBKdSOvq0mgaq/rnsYN/CQl8kAL5O92rAiYqL8BI16xNZjYH2UtfSR7gs6B+mcbxMWMRG/goG2x9O/tlXS4qeQXOxJHCHk5bBDKqX85kZOlSP8JQwxtgqlHWLtj8oIvQQbEzobI+tC6F/3ALZilLzFJ6G6ZaZi4jQ/3aOzVcGD4CHrlYMZqdNMy3JstDGmOgs6rOCWqIWK7skbpAr1Hhda8Ozh3q+Q5okYJekjftvLfvWCLk/xftLLzcss8AbNtwuYTQbxxqsd8Hg9HVvB60A2NpwLc/bEOCoy3vi1CEoNAWLAqGO2IIlWNI2RNv4gkFKwp5LBPX7jP9vAzcCpmItBikMKXH4ooPMbD4X+i2fBMgIr4b/7yLuNA958HOk90G5Vig3li67HC0s+xwecH/hsJSKqVYDf6QyXJfUXpxBYRP1QDpccQSty8H3SEwIQ4+BjCn1P6ZFVSbLcKJfRtykkDi1YbWpriqV0pequfzg//wro2lvF2AoBnL/D9xztsW+cH6+Cp9lu55HL5fh64+KqbyW5dNFZG20tvMBfYYhYKxRhISE5BUaZI4s2G023MPeH29ZwCRHbnyIaRxLLlcJUGyvgxwIZuX3dlJTqRMSNc4loVISlMes+K+/91FaPKxOaojh6hAfGF5Bw5WvkNnhNOgyim/DBlPAEEROHKcirCbhpid549nBvcJnrNw0UJ+wbueRxZfZAd0Z6fod7/Z29GBl7Bn4mTY3nNBmraohJhMd42j2NrXjD4hz0afcqBuVnP4gz777z/wHX+8yn", "online.py": "eNq9fWtzE9e24PdU5T/0JVVX3SAUG0gmcY4zN5lLmDMXklSSO1RdH4+qLbVtjaWWom5jTIYqAzGWjR8kMRhjEzDhFRLbJAHjJ666P+D8iBx1S/qUvzBrrb139+6HZJPcc6hzYqm1H2uvvfZ679UHDhz4108//lCp3fzSWXxU29x1lh9UN+78beRCY2zKnX3h/nSnuj5du7VUW7zjPP/eWdj8bbtS+/HH6vpIbbzy2/ZkfeVFfWUJnihHodMxpT5x8bft8b+NXHz1lVdf4UPOXHSvPYF+1c3Jxsg4dIV+vbplH87kDd1U3j+muKuz9ReXnKvTtWvz8Ht1/TEM8tv2zVdfOayU9LJesDoT+WJZT8Az5+qXzk8XlVMnP0aoqzvTtYffOxdmag+3FGyS1l+nPz0wYULP5xOKM/PAmbijlA09Wxy0AcqenG7R+JPOzFR1+45za865POreXXU2HxDk/qRGqVws+bO6o3cbIzcbF75RMjh5/cU3zuh9QFBj7hFM9z6MfQLHT+swfG32kWIcxv6KQOrFjOLeWHVWN+oPv26MfO1OP0IMPvvSvbNd39149RVFMZRfL19V9F8n5g3lkPJr5WImbb8Of/66DuM7O9+4V6Zg6PrzRwjz1lptaxlGqC2PVzen3Z933bm1xtxTtigAr/bzi9rSClvXb9sLDIrq7pIz81z561OlE8c/iaNnAEePnZnrYoJKRoF9ra5vOjNrzvJVmAJhq65P1bZmqusTgAgYHVDgLHwLMzFUugtP3etPYNC/fgeD4wJgvdDFWb7hTF92Zn4GxOBXAo4BChP9x2Am02/YCixBOdJ25Cgbq7o+7izcgZnrD790KvO4FCQ5/hBQSJuJO6D06bYB48DUzv0xtjZBf9X1Zffbuzjn5Sl3cZwTI1EK/e5eH3MXf3SfXXAnx5VM/6A5oDj3vqxdvYwonVhzRy7IeK7uXFayhq1n+oEa3NUZ/GG0Ul950rj7pfvsijt127n3E5LU5Snc4NX70KC6ec+ZGa9dfsZWDZTt/rhEcx84cODVV3KFUrFsK5liaTipFHS733tkDhZKw4puKWbJe2YXy5n+JPuTMs1U76CZsXNFU89jww9efaW3XCwoWatkpm3jrK3wblZGN/EsvvpK1uhV0mwJ6lmtA7dUUXK9Ss7KmZatmxlDPSvG/8wwrWJZNMJ/ZcMeLJvK2RQfQWvSX7UHS3kjqeRzlq3FDGAPl6CdpgpIcprSWywrOSUHg/NBxVzK3v9eU9hWNcYmgRzca0+B1VR3pmo7K3hgnt+v7izWvt8EYgPyqz2ec2a+c3955H4540wsAJ1IiMllLTWr23oSdvlMLmPEI4i1sOwAavAhED7+SRlmppg11OBCGE5xf9K0s6pZom89g729RlntGbYNNrkGsyOCOqHBYM6039JSwCcRY/AAvr95TNNSdlHlIAb2FZ+oBZg8L2Djs9OzlFHoSQ0Zub5+O8XaSp3NfJ71TCpsgXQacJ1wujo/LJq4nwAIfRSjAwnT8Uam4iw+qa9uIX2vPHOW50AIAJU3Rq6yz94xqG5ccb6ehI0xdZtO6QV2eIDfw7bBV3fxB7adyK5uzcG2IRfFqZX6EjDaifqLjdq1STymN79kfWEiOky0V1kL9kHayiBa+KbYRTupmNCwLdWWVNrYw6Gc3S/OVjHdV9azqrzFHomWdbPPUKFf3jBVmEjjyJIb47+e/ADMAA26ch05YOXUqDvYxjI+hzbYEoiMFpmzFESxYuQtg0OT0W21C39MYstuLTgE9aIxug63d4TGx1EBSphGU/6kHAlBiP8yRdPOmYNGaNA+vvUwMmEOh+hiZNBxuL2b/xoCBdCqHOpUPkhlykXLShumDaJvWM33dbVhDxgBAEwCUWYHiW91JqzBQkJL5WyjoGrvwI5Abw/ew0p76AjhpvGZJdLNFMtwdgJkD+TApDXyhMXb7vJ3yBkeLSHH/gm0k8pJfdgoMwWj/ny1NvsT+xF0HeWTU6RVwEeU9hMT/JfZFyA9lBROpsCoMs1x+Lr6gJ3ZdlmFM5TAdgkgEcba8kg37BDmcWarW4K/VCzmo/AHwEDxhuLriXMLNKNppYSbngPpuz3ijP7CpB0pce7PS9AwDrh8PCBIIf265cHNR05oMoTAqM4ZZhqlVCx7wR+ApRglqUWqbHw+mIOdoZOUVj/QgaA5u/qX0CkTDAxEekG1DZAJRjmpgN5UKNkW7TkQSKmzPdUmIag+csFZr1S3pmGn3ZmrtXuol7rjG4ANZwzUlq/qY49hY2uzt93KVSAt4DCMF6HCtTsLiovCuh3iTVABpgbO9p3G2IyMRHYc4Cxw4FQOG6d/0Cnhty7+kB9BRHfa5xemzB7MszQYYaEwmIcDWCzk9Lz6Qcoq9toF/SyemY6kAidNeZ0Wj5+B0bRLJw5mTemlkmFmVRhP+iECLfyMT+IkEvIWGIiN7G14QbcGjGzaso2SWizZqJrQA6YTJ5UB2Ov0GSOTVM4Z5SJ/3KlqAcFwH3Ra0Kv4+Zn+vnbngjv+sL40Wd2+4Uxed5fvoxq1+JgMgmXejDRVEN1KG8qC5ftgEDjTt907z90Ld517U8yQcDbW3PXRxsKaM7qmvJfVC6dhd0caN9aciUewc6iKrlUa8/fYVzgw7q1LYMzUl544YzPu9ow7uyHvLq2mB/AllpXqgSMpBHipbJxJF5Ls7zncZ2BhXd37lhklOnIy/kJcuCBm7kmdyRlD6uH2pHKwC3b+oKKWUtkcsEbkhZoWYe2lFM6IIsMEzovcOYbBs0ZIZmm1EBqDrU1QkQpUhgtNkRqTycN4sKVazHKkXe/4Q1DRQGocVOdkqOJBAspMEYlq+98MWmExn8VVsNWHgBNTASdLq2ywIThDoEAIMKh/eDfovDGAQDqlQBSopeiGQSvdzCoJ42wprZ/pSyAUlh2DHsvu8hp1R/eOL0Vex7lW60CA/4HwersqGEpPKSO0SzRM0qTucrWp88gbb8p8Y2saDjY3vla3nJVJ1OhXnpFNyHVNj4OjEUYs212drY1vkAbpn2uhMKRRI5QU3AgIYcZoA9M14f9okKXyxT71CF9LBqSjpXxk5nOmcdLQy6ZR5pAT20znzJydToP6ku9NKny2fLmz3Th8VKz2zWNAR8BeGACdb77xxtE3kbnkiuWcPdz5WXnQQFXJyHa2JaN4DrpCUJUCgT3cSWrsUJb91fP5Yga1dvhsFs8Y+XRZB20LxOeRN6CVXi6kCRSrs/1NNPZKoIFYnQCWBYMbaXSNdB5uSx3hI6VxHZ0J0xhKxAHUWzaMcwboeplOEvFimGKfnjM7aRir0JseQGjwg1GyEB/wuKeP40A2KTgVhDw+lcbWjfrKPaawOTOPmadBdvww8qjdfeKM/lTd/AEpZwwVguNH3kYpM7vBnrg3VhsX5pzdG9X1LWeB+S0WmI+ovvstawOipP7sOXSBnxh+mbkPhjub133+3Bm9T36x5dr4GBAnKmpk/TNBAx391QxlobsMqnP1K6WNdDtJKgGop/78yXufgko3U929Vbs2D0qrU5knUv+2tgUTVZgPrrp+pf7kEvOSCID5IORXiNkYJCpE3PoDABoF7bdXmCvEnVtzLm/Vtr6pbV8H8Uya5mTjwq4zOsWewKFyXlxwty8C3LU7T51bM87CprPClWZYp7TZiKXrT5yr8MsjMrh/ZjId9arKLefhFefut7WFCVjpu21Kdes+TOJWHrvXl4V7ajwAP1HNu20w6qenPkC01BbW0a1yadKdXwWdgQ2Osp55YgRVKJnBMmisxuevq0Bh+OEQpztNcRdHGj/cQKcRjY46IRtGZhvMLsv3pvgBps/CHMbP+bKwijwzOV8O9fWPOO8kTjh0lX8Tj0PdGd3xrkNZ6CSeDGVDTf195s2lLYF+gZ/9X0KDEDqS/mfAFv/GcQgjiTbiZzy2YbDNPmhollKg82aLBbAJenVQb9PwXEWepsVgiTSqZp6dLhWVxdFKbfNhUnE2Z5yZ5/CXaenwrTKndcch3sqdM8TGFQdNlHW+nS/tb9nMEcQxFg75leZ3nHvfc//G+BS3Ib+9gmfpwk337gOgSrAy6l+vNx7MoDOYTBJGksAt2HkFIwO9H1enGXnGKYdkjxHLM2ywyVQtIsubGFOB5ZCjGsmLeKfS2alw37XfDqSXgb5FkHCq1AdlPKMvMDES3GON6xF+aVDjnalr7q0l99kFz+cMbAc4IQ8SjG9I05BFi2iVTdt9QUEdJCDcm+hAdmdX3ckLIUMd+Mhp/YzxoYHS+mPoRza94jxBm8Kdvo9e+pGr7i936ytPnJ1rwF4kCMlngJSXZxa9ZxsTAN2o+6j+yZFAY36ZgM/BHxU1Jm89oT3MF4fIRvSnYjB4VKDmU+8DI0mdwP/0pHWZso086tX+vhKRF/KlhA/ZntN1laS1luTFyiCcbqfp6b+njwRhsIzfsSYWkMHhWEwmcGD7gR8hc0NPMcNmisdoAsdBUw4pXexn1E66AxjnaMHZhFbE6T4pISp8pLypuyIMhMYjrcc7TKEmfeXiYInI54sEa5DoQGTAhMy/ms4aGX0YHg5lzycVqRGbNtoOFLfzYUCKpGYidgDNh3jXGPYhNd6Lb+CR8ylUi7Bb/+gzmeyOX6mtzLGzXl/5XtEH7SIZd6FIUxjyku/egM+5QoqMdFVCHunF+bJ0eF4jzxuL4tQfPnBmvqKIJLq8nMrlxugUCzsqR46mjmKMUfnoCNMH3dtXQQVj+ld9bbS+O1bf/cpdvO3cGnNGtlHYS1qw4oxMAeScR/9y15m57inMpA4sjjhPvkINRjgMeehLAOlOTHCgLs+7lav11fvo8F58VN/FsCWMiFBv/AKTu7fv11fvgo7kLiP7l4OrqKyASnr3B7YyT08KarQVTyurrk87o2vVnXnWEg2hiUcRhQmxK9bCpZ+0cqFM+Po/7JHfPNBSahQan5sLAVUDz2RSml6cHb+t1ExqgeiUTQyrlM/ZGGk98eEJVPgAOfXv0PeJjp/KWq2yJnZkPARVntzNBV2smsOeDOIkGwAS7DqAkbm025L8Q4QboLbTn7Mtj57RtrVU4pxdbd2pD0WwqNN7pGezZR5g0YhBkTQkuRFUmUDB+PHH6uYkhdFvgkhrjE26i+Oy7gyL1zN27gyiE1ThcUa6TMd2Vi5Vt36sj5AgvrEKBOXOPfRVbiJSpD4wWLyHo48alx5xJxqPzK/WV5ZZ+LQ2e7t2H2yNGWeU2xeM6Bi5hRFDe8p3mvxl3jHyZBJ98nElVgJL0VK6OaxqMbLTHzfMsQ0D9dkvclm1pMVyv/PBDsBoJPFkJll7DxoUVSkTZAx3y3FZg4AwOUIKCvQwaRlsWv4IYYlGWrCJGJ2U7ED7GK8JQijcXKGewcbearFHrDctxOjR4ou6c5r0ltiygEYSWdgpXmBp7wQE1aFOattk/N8lE7gtZBl22jIsK1c01SDJCB+C7G2k7WL23btKW4SKyBYr9ubywJH6VDEAOXiwQTQ2Idw7Ui/m4BF9YxxZ3I1R27pdX1nCGM7lUQxTsAgUCAsyYuG41leX3etwwCY8qcG84Y27X9Wmx1DwvlhQ0P6NNX6rO7vAB8K2a0j79k3ZIOoQXXGaI/OCBYfjRivSpzSGH2r1MSHCrdKs0rQZ045hpCEmGmCgfl/y5lnclAsy0iHdSttlZu/6E6foERh4cWZ9yjij51VtP+FddL43CfFaRt+eIV4vBNv3ciFYQkVGL8FuGCE9Ff+lOTL8JeEUoRBtiAXAaFYyPLD8NYxYL45FkjOJfymkhQPFLcSk+C80TVmDBRWRGAlb8BixhxIvystx5e1l3MGV9lUNswnfUYFgoPP2rEo00+6dbS/cmg7EY4hYQyeXnVkvfMV1zzaltvW0ur6pfIiyePprZ/2SnzbHpGplrr70iB1cHgJjka/l+3/H0xqgjqYHlpkspVjZyQI3YGOUjK6O9m4U0uJsasRQUV6XkM0W2AAFGoEMN0ZBKE4jRhv9RBKtWzb8ONQlK7AzmWKhNGgbaeTduEGczSKxhXYHOCJG7leWwDqROaP7dKv+/BIYLooqvIBACcIPCAChF7CFG5DtlaLCov/oPimAowBdisQO8ZxN/afOiIjiyAmexKabCuv8/Sw1lyErADEMg3a1d3MKaMPgY+RpO/lF8DHTZtuD3HMfTAKBbcUg2GLwv/z8IoDBqHfOBDVSsC1JowzsgshowEagu+SsNLOeI24AMRr7+/84zXvtUYHQAhays77e+H6LBRk4zTxZr26MMjc6D1LNoMFGYjvs87PIA9NJ5ImUGaCQQwH/q6alOJcCrSOvEoSYBWC8HRCK6PGjUdE0G1ALOVOVPbqMGGCsnJmFzZfMOQw5/27Kwc5dMCEwCqU91Rajxgu7qXkYONAO50mrOKwWYRP4NMAomAFHoV+fRYDABisQA1oDcRElFkvk+WlXp9GFW7mHwQdyOKCtsztaf3ABIyJkkwv7qYJ5sfSE5W4yqxsmoeSFH2E0L8lNJIBWPLPOmbkOvMa9NFFfWUNBMXYHWA3LTUT/AjkIqjuLCAOfbjw2EiEMWjw80qkjm13oAMIGjqrCgRHejboBsBH2Fr/J7pKDwcGb2N3kpLIsb/dlWPxstrbU22+EBwSiB1MCH+OTCCtlKxQs0zPkW1BVE/U1yraDXLS/aJF5KQvSPfIJBN8b4nxvCI5nuYV2FGPhNklnlbwC0XAa5kUyB5REXCGaYq4v2fh31h/Un95GzxKmxU4iUS8+4o1fVNBhNrfmjk/Vn92NifFikkqRtH2V8YghnXHdIU35Z84zZeNeA3yZyFJUDbOYQIZ0BPavO34OrinjVBoN1BHfrrnOHGSLcEhDTNHzSOE88LMWxxgD1IT65AewnHJBz+fOGWq5q41i5t2p3nxRt1Vg0oZuqm2YtpsrdB5u15quLeIg61S4y6vD2/qQ54tzGNwqYBFgQ65sgK3oju+6V0Z5wHlxxLl3E3kLEQrf7J1vahXycV57Wt3a4e6zrfkgGU16ZASNMUkKaIUIAh2R68u1ixvN0d+TBz0P6WHIRp8FxwWcYkAqbjkezG6NhHybRolMQbKRegTIB+Wt+K357HCASGbbKsGR0st9qC1oLbrgQc3wYJsFdECKq9aCwEQn9BuVC8Jz1MxrtMdALKup0JVBeYkfoG936y6e8GWdZBr0f8JhgIOWP9+D+gKDoh+WQ+J95QOFnLrvKL7+LvVg3/e5BNjLLDJ6qb/3qPUQEl1g8MciXaMFR2As/FBncIvf2YNV+MgUc5Q/b9UyBM87EhLFw1ZYjGkTisf/wTUJRh0+cCGOHCue9sV9m09dAm2dZUZYXcGNplXjMbWKZRtBCIqDGGgkGodRu6Nc+GUYsI9GHMxDY5PQSdbTim26AqOy7vtRi4PhBd+ZSooWhv+C0YkE6jwUOkTFNQGYEm38gEYCcDeYN7AZg+O8Fms0c8EizOaQ9st1Wqb3zjyuXZuPBp+Y8YtyYSOcFRS8xbfAwnN8TFAaWDY18nzMuHrrjRg1NiD/hArXVH3j0Rh5i/anx/1xoyYIZcS+YZmvVijszfMBJKjoOAk11ZMcrSPjzAcT8OR3xxpDIu064D4JusnDNMD4je1HBtl9EqYVlfUMZUMwrPv/DRAahYwJKItbXWUjE3bM1GZ/YhcvWNIFuljUQbw5BDYaKE4ZTUHLOMMyhGsrc4p0YdIPO8tXIuULeM2Iii8gnqKCSyQHKXo3kNfRAkIuYKA6uhmRDAGu4a4Z5mDBAMPIUM/lStJ288sUHCVhbQAa0O0wvmz0oURSpOX7PkS36XxuwFAzIW6WRWcFTZZ6P0X3xJSX/fea4uEWg4h0pms/vyC0V3pLR48Ewqv4T+ChF+YepLtntoaPvI8Z70YgfxTsT+ahnjLOlsJKXV+Pd1iNnInaYqLHLgwmC4PZw+/Cp2wiyWanBYe92Zj37PWnS1KZNg6Ap4Mi2PgFPeLdQU8SEQfGwFEJbVcOKzqpo2zYwwCbtk98+nd0z6VtQOI5vOLbmMMUAmDtfTmbB1Ppam3ojlhS+SypnEoq/064JT9bNAvbp+CuXHcTMifmRIgu9GRhSVZ/R4AN9tA08NjjhIOcA4oLh5F9C50enBsUi3Pqv4sdycLeR76Kz6HBjPfhGBkn8D89bUIB8gYOX2l7g1ZCG0f/o71ECtJBnvpI82cOTwbTvP8ZucZAOYDxgI66UqkUH8nf8dAU2I08fq8pxv95P20TeeCnL+xfK1+3nwcdGTrCRySWwdCkJ3CBLSe1otOdkB2MOOsJb9YT0qxWZLYexKQ0GzBzwq3WlP3xDQTEEKDUP8xf8BIrsqvYgywYDzGvuIObTeKhPfwufEokaReysUOc2PcQJ5oNAQKj6SDeGDREDxtC9h9ms1yEWcbnzG8Y1pcov5tlZbhbS+gcAet45ZmXjdr44QZeuF8cqe5M10dG3fEpd2HXnbpL2RmToBrBZ/TOzD1EFyIlrlJq0Uw4awddNZMjzupGbfQh8OPGzRl3cdObBlPHKRtcGRDZSfce4mX4Kz/WfrjCGjdmV3gFgK3nABMbsP7Nl+6TpwB9JI4R9tV5ickkgZhSLZJ5eR5vwLKWsmqD3ki8xUmWh5x5SyiWeg/15/JGKFMXyNu79PluOH06knKIFobw5zD//oAx3MnZng1Mr6vjSHfMXSkMteSLoC3GMM6w0zCQTQwaWaF4xlDzRZHw4EN+mEUyYdzAnH5fbgIgbqK9pduuQU1OF/QpX1XBOgfyHUjfsc292qC2SzlmoKf3UVLbxg14zJ7Vpseq6xORYgmTyJAxJ236UX13gd29YZdt4jR5P+wvQ9c68B974XqvW9V+4v1eV6v9lq3uV3t6b9NL1pJmHHPT2u//X33dulXisMBS3ovYyipnEyMdiTYTkzVAho2QvoGErVZxTX4dw3NbiNwaP7wgBTkDsYe9RoWFM5DQxV/ojVlNrMHXIi/Cu/waSIwI3V33jKGwaEHGuNcF9pdeAotO7bWQlr/HJWaIiePywYDnsJsNqh8a49fsky9zP9+EWROap943yeXwB0fFSr7A3/xSvzSGnucyKf5QsJsJPq0hnjri9HD5XhYIXnY1C4sOSVcKeaWaiWfuGkb2aotXPM96Y+SOW7mKVSXohlhT/5YnHDP9ReR4bG0cMGRbwNo7UUQdSyrB3zR+YQ+U7/ANDvHv7LCn0ASLhQC5ZgY4h8LBQJkDmePzT3KTNakVAowy6u3wZF+e3ZCUzsvZYWEyxbQXG0YKtLRBGo/hCXbMt+1ghNzKyJzI0qGrzvCVpRfgdEmFzQ10KDXSYtIh+8pSCkuJ3xdIs0TC2FTLUlciX054vtd8GUDro0oScbmK7A6rnAK4X15N+En1wG4N6eVsXBgDnQTs7gl8wlyL5uy9+79QFkQ5iLfggHMn5BVpcvdj76XSnXZ0SCYlr1nUsecl16RZ9sc+REZUenkDxEAZCwdbdozzNGa2WKZOQpB5G4NjedlLlLfUAhxxpQ4HgWn85wHtRPWfH/Tax6i3HgYRHXL+Cz0lBi/nwUip2fz3UNYLS9COW3i/WHez7izBW0T44rNm2NW6YO5MxZn6pfpi11187GyPKJTjNb4H/vpfDn39TbBHivK7sSGP8NCeegPHkJNTk3MXqKIh+ApXPuKyELWkPFOzaSWVFb5awT3w3dwSy/fIPoYYmxzsACeUSyxEdI/9+hx5rIGq6PDb2xtXeDxi5ZIzfdtZfOLObigUDPvbyAUWmwaj1i/RByb27EZ1+ya7h8yS/5yFF81YmpQkdJDrQ3H6vKhyJIqRRZJmZZ2+s1MyNUDQtXd468MbPWvu1lLj2i5WMLn5JatZwCom8uSKSzugYLjjG01BJr/E56lMaVDVUkz0awHlLZoc9UWip5RJ9xggFoz0YCkLq0l0xNcwoKQa/yIKMASy3TAAxW2u83tnwafLRi9KZqLguFATOpV3vsHdoruyjbEp2CeWThu6LF/dWayubyIpSO4O5/nPaKtuY04Da19f+b66M4W4hOfrU+wnZ/6x890T9tUZW2Z6H6AZ1bcfbvDcCMZdpu5A95jQgXcZzvOAYLjfe4haZuvUoj11NaGgWZ64xlGbaWmtLWi0a/ajpjU1YdAJaFqlomWooHUdkVWtOJWfoqrtWrNgqbi5jUWj8GY5r3MFoj0tHmCdmFxJZcFNXwnuDrjiCka5z0hjACxCS/tIvyrpubLFwOA3V6NBtRZBuu6oWnk6qUDvHtKocfAY5ngaDRpd+RelJ/pbT3xFGzK9AzVl0LFmshiProljan1etlVdqMNHurWAW8jKG17CI6hrPUa6mM+Kj6aBaWg2Vtc40tZGN2mOGofBAunR7Ux/51tULCYNm4k1QrC0hwnWS5kVDKG6W+yjyYpwYZWS8AqyQAa5fJ530Yd5UcAM+aSSsFsm1TaBafgvZWMoVzZERRPQXGgB9D3iba24ExPy5UdWtxQ9WnCwR++KshoVvOrGsyzHeaXNxUdSndqJ+soyQ4PiLt8POFrZiqmgBNaC/beTKruu/+vIdWAstdnbGoYd6RF6w2hqEFBeHdxgAYBJb17GpPAKD96UrRw/2v7b9kJ15zIbCtnZ+gOm7XimJXQ/fvSIwkbEewVvKfWdFffGi9q9zcbszfrahLO9iRYspmNdxq/PV8J+Y9ontppIKQIAgxXlbdxYA9iUNzClUGlXGktYHe3XkVn4nzv3vTv/SCynEip1AM3AapZLnGCpuXtPgzXVGLEo8OuvI4sAw68jtwK3ZVHW332A5WIB4dT2P58D3uWpA9iv7t6q7szAwkFbcNYvYf4ibSompIkCkvWnt+tPv4M+nt0eRAujUkIM9Kyug30/5mHFGZlvjIwr7f/NxE361DTyeVF39oiHmSgNePuOWwlaiVScCEUd4I3dq8Sir7fwbFBV5Cc4+eqMnPHL6lUGysOw1eNsVyk7meaRPRbMS4Hnixe+QJqhMqaMCqGfe+c5mwJRvfyALYDKriAaAT8MysatOx5KgOU4uz9glH73MpBZ48I3hOQrWGFh+V59d4EduQBu4WhjRETcWhPV9BZqW/PVzct4H3wWlLmbcLQU0NRw80VTXDrFKhjg3mER+9Gu/OfDo6mjLDmFdXSvPXGncEHulXEgCnTa+CmQU87MHEyGw97YcieXwocjY7DQDS5XEI5XcAdPx4UJtD5AK6NLt6i8LWK1ZrfyHMterN6iDVzGG7YwDqsB+e0VdxqQ/RWQunNphjVuXPwOcF9ffQpdGPEfP3aM8i5H2RRYu7QtrrIN8UJcLxyrq6v8Hgq73ctLbQMiSHdkJ7N+9we6UMCuB8qX06vrs5hTIVWrwAJGY09DKTzYZghV66AeVAioFP5zNOoH0cgL1XZk+nI6m8vYAW2U3SJKe926Sn5OAC/bFnvvSBLBvXm9jwlzqvgWyIOROhcCt4y6A+SJEswYwmgGVRyT5CSoO4FH0Cys0si6iNZEU0StBy1nfMJZTbwr8qc7te83aWumBPch2iaWEi7FZpBLQoqKFKUgCwl5r4qtRlo8FX4LG4VB/w60EgCGQryBErnDgbGDLfehgLHax8PpPPqaCyqOLXna0YON/i0594SXyMFiVDIfRdZ37SkQehgEk8f74BNzp2bFnUw5vMQKXUYq+RCymOCPw9EQK1akl8v6MKrrbfHq+jvUcAhY5hBlNGtNYqfRDSDxSI5r3OOOvc1kFJcgd6fu1pcmZVnLqKe69V196ZGnGJC28SYKuqCorm0tO5fnkXNdm+dya/eGOz0ODLS+OhVxqvCypb4tM5jPqyoX7pgyA/9vE0kirAkc6D7P1dCUfmgAojVW2VUuuiSVd5XCfIHaCIINAiOWVMCFQCm1yk1WCBTZ5rV5THGnuxNARyj1nk3CYYNt88qkUdWMFtXgWLm48YCAfk0osuSyV7ySabwcEYGlnMbbXKf7oGOk7FqFlV1jxRtQWEdqrrGMBiuFJU+R4Imd95S8UmzsHQNMmDcrxRYQMOH6SzKnG8iR6y6BxcCxQo6ot4OfMVsZ/2KibSoRU5whriRSON2OTUB/DmFJHr42mgBWR39hdYHxrUAloYGk4PTRRPtYB12wXsOAx6EHUFqVbQsZmYoQcbcn21Bh7g+kDDPL2qiJ1OkyApg63ZfQNFm6+MIrHymisFf5BSFCoj3jeH9soQTVovpCrEAC7QeiLIbtkGKRZlmiLNOA4TZa30R4jTuivvjYm6b4r19HLy/83PyGpe9ZjvUoR5xrqqcO+SE+L8Ks4SaxSZvcqGrRuZlj08PQ784Ujr3WFXPjMQgU3XNkAuTdlpebBtLsVmr4MifeeFGDkxFiaUzhIWxxBcRfeVcQKLolRbN696Eimc/7cRlHMev5qF8OsewOb/yuR2aSvhyMUN6+ghGsUB9jz3eey1GJ+JsJaaZDhWvRvMRt+PNNBg5yQo/hiPIymriIRBAQD25yaV++98+rxZNqDae1lE/xR1qkGFyI4eX5XW7evmMv/eWTUx+jqiIK1Tu3xupLS+zWIbPsRA3TZRYECNX0YrI3LG95n43b4UJEbHEpf32sZ+haup9dRNxW64jN+WgL07qkQO8jvEFWvGy2g83uW+nCJgdVwBl9BNqJZ4d7VnpMTEDONYCjb/QhJbUFtWBSiynjgBx9cRkF3lnkbumsnzcFvaUUAis23J+P9Cd9v9kIcNRAMACEbdFsa3ZyyVI4q7ErPC3b8q3hWQV8NyijYCAPJugZ9QOMbKRFzXwrz6vkBx/b4jE+BH2gz7C5U1LyehP20OediIl0MQMCT1vI8OoIkQE6QCS3B+Xa3eS0AbQL5+DBFlCzFxxxVjc8V4fnHorioRjegmLTLWxGL8Um9BK35fu0+3zq4K8ZKO5nT/n+F19+//k2/MO3P2j9AwRh61uom5k4uRjKJXmHRfNapYfkemWJ1rqMfiTOTCFmT5IkpZFeLvrbLPDbol7RS2RHEboBzs6h0PB/l/SnpgaqiKkV5FBaUx4vc3c4qbVLy+HRkFb7AslN8NVPboqVO5y4M0ZMphTCt598KKAY4QFpTS6/92RL6MFju88VvvwRxgX/PY4w+R/iXFd43YG8FF0RogsTsucQkbgnu+9EA/jZFoimP4562VmiZl+Cs2Z/h2QlP9U/lrG+HF98GZ74+/hhE14Yxwf52zd6ke+Q03ovd0CvFlWre2W1uoVK2xtwcLeTg7v9pR3cRWTN1PNPncxFfkjBt1IIpQZzOdF/jj/zQDHnethK9l1TJeDl2tYlUMyr6/fqI6PO5mxtYZ2l6uB7C56t1C5u1FdXnecrERdIcSD88ht2SVyKKrAQwu9zRfOd6RG5B1KgIRkMUTQ7eyUepe/RWla+blk/Xov17F5/QnU1RlD735pmsZ1AbHF7xHl4JWaaYMZNk2zIfdRbitZciuQqvFSlTp5Gy41q5o99/jMPmu58TV7Zilu5jg8XvnW3bvLc6zm8wMRfzbSz6Gze3m+9Ty/jKVMsFHLA3rKJDqCopJJAkmUpUPiE4kD0TO8F65s9ake/pjHkN8NTwJ+JZnhAzkuvwbL0QilviNfGMIe1/zqyttRb3hsis1a4aEAXcwZQH++VkFp31GsuJ/rwFbK3QXKvOZ8dpvCmJg4vEnMwO59GTwzavYffwhtv5XKxbHUmeK55gr+85sP3Th3/NP3Z8ff+x/9EV0Dis2IBXZ0nc/lh/HtKP4t/3jNNKqH+vmHS0xx9+1Sntv9RNKiLUcQ/x8/oiW5/3E8/o2E/NPDWXeKTQbsf/35U0Mmj+ues1PjjTz56/zi1/rccDfy/ijTbxzBbd7AWEtn+jceT7uxabZYKK65Pu9fmwajCF4Q9u1JfveZcnXKW5xqXdoA3NcamnJk7r75CywSgTn188j0ADefirxj6wjwPfGIQSxoqPXlQGw3yGtv9htIHkscwU8p7/g8WtLIKOpid6EzqwYQBu1+3lTIcfEsZ6jdMZbg4iMXZBwwlZ6cUHN7qLwLDxRHFOFRB2CY3ROov5gGeXHPgLwdO42A0C2/63/9yQCH5hSOllL8caAlMElpbei6rnCoWsPGfm0P2lwPSxAjlkG5B32xK+QzbxsKdpC+hKQOLYf0KubyRlRd2/IxRHm4FOOvYr0srTymnfUAAaOBlHAAxDi2NT9ONe/zpZ7Fb/BEYZVl9mIayw1thFxULVLpinhYe3lxcFENeE4Tg4g1cHVp+0YVjL9BkbLkTbswpDxv/BFuWKefEBoe2N4jhZlspMCA4VdYoFFHip4eKoDuJ97iK2936MH/RUhvXf3iC1sGDefYOK+n1W17IkL2DuzZ7uzFyp746JaCkyzP3na8wz8QZn3rdu3EIouz16vbN+uotzPjcWcHv8kvD4Tv7AQUeHVvUVzDW5XlA8IEXx/KgfreNYm6jFTaVd93RnV9pjNx0r41WN69QZpg/kDt9t/F1xeMGPFfs0gxWWF5f98QwBtiWNqsvdp2lH5zRUS+NLPA+WfYKZe+dyuwpr0MrvYXXi0IDGSYSqf9bzJmqnerFijW2anaazNNqIqeR+TA+JPYT4leBO+HH2ujllTM3qjvfiJnoits+JwLGLM0jnxkOuPRCFhC97A2cKSpNhhoTea//mf9i2cEf2ABn6EaQhySm0zEplrBz5nBC6xI2ETpPOrmEg25dHcfa3n5TWI4n4afA29UELfukKgQl+d8T/CnI7i8GOpQzInx4hrzx7DeKi/DC7gPKPwHWRLXqxHkQOR6dwRjeZ3iOS/WVhcAL7CyseRHUOaTfURc/77042htSUsZO0ktOVB8Hb7W/faQD/4O2mOjQLVv5J1tpRVaXr+ykoT9dcgpBxPHLZqYN1vZ6SR7egj3yxptJ6MULx5yk26/7KYsB+i7JaM/PXd3CTE3vRd3eLnYxRDPdnABv8vq9Fljwh6Gfw8unDZNaIq6k+WIRRQ2ZrYhVq9i1MQozAoF9cV68GZQbWvs9ikztaXXmX8M3+GL2HXKhC/iZbK367lgYLgIJdoaSgn1SwtPU8Wb7sWN4l05KDT54UJXg1zTZaU3KleCZ04y9szegIEYADmI3+Hq9hfXq5r0w0hma9o16H/oWmCfFm6G9S3qfbglModxZepsRqq9hPY70V0mecgWWCeR+YBPy3X9KkAhr+Dg8UH1bW/iAeRB5ryVFJiHmJdd7F7o62RBaR3dIpS8bFhPXOXxPJOY1pNNUATKdLug5M51OdLQUOAH22jOYy2fVBD4g5TzB/FDe7/96/H/z2aPmtUjgATbgd/jko48+gwOXKA+a1uv4+PCnqRKis6CX0nTNDp05wQk0f1sExw2pIUkl2ENwYmkTSmWMaWN/GO7/A/VpDUA=", "tools/train_rmp_lm.py": "eNrtXP9zFMeV/11/RR+p3M6Y1WhXCEWoMlRMDFyqjM0Z3bnqdKqp0W6vNNHszHhmVkKnKCViA5IRoMR8SQDHQGxHsS3AcWIDAvO/XDS70k/8C/ded89Mz+yskO347n4IZWtn+svr1+/Lp1+/7t19+/Z1LnzW+fRC+8rX0Yd/7KwskzdOnCTb9ze215fa63ei3194/mT5TXOWvkZDsp8cOXn0+ZOVnbvvbN/b6Gy+vfXw0s758+3bX22/c6N96zNouvP+3Wjt1+Snrm1OkpO+S16uVipkgLw6BP3+tvSrvr7OyqPOrx49f3Kjr6qR6N5qdHZ9+53POx9vRh9e23l7nbz66onnT1Y7N//SvvQR54p4rmtr3jzp3HiHCF5gqONmSOvkFWqHJmeuffbuztKNnTPvwXP04Y3owePo8ccwleju29tf3e9c+bz9+W3g4vmTm32DGtl6+qxzZZ0cPznWP8jntRxaM6E7Qx0Yf9atmZP6wcqByhDrskrdYCCgnmHVoXTw4DDZPv9J9O56dGu9fe18+84ftr9+1Lm62n54tvOnx9HXn3Qu3e+8+2V76QyOdkAj7b+eiZ5cjt69vf30KTlmOfRNOtl/tN6CUQOz6dm0v1o5MgZDAetrl6LLDzqPn3XWL4Cko8vX2ivvRatngYmthxeji7c7fzjTufnbvy2dgdf2hWswbSZlHGlII5ON6jAIoGGbZPv3F6O15ejc2fbthyif1ZX2F3fJlG/WLeqERm2a1mY813JCy5mCet6cvFw3m2+iCC99EG2sta9tgLo6mxugca6/K+vtL66i/gjx5sNp1yEh6CcYCH3Tcgy/6Rl2E7XF3kl/v+fTAPTT5G9uKyQDNdcJgYOBum/N0oET86+wT+xaPTDShFZMDQEZrMT/XjQYF+KLKA/YYDNBqHkh48ttwsO+150aJS0PSJsktJp0X9++ffv6rKbn+iEx/SnP9ANaJj8PXKcMswiny8QNyuStFm1BcTAPz+G0T0GozlSZUehrAGniQVPbmiSC0kl4jamGrl/LvmiOozVaTi20XMe0iRmQY319b7z++hjRWU/FMBqWTQ1D1UCcrj1LFVUDzmCu4qPPapAg9BXspBLHDQnIG7jTkI9RkB9J3jTLCagfKpVy2kPtE1yDeGOWha/19R19fcz42SvACjP9vn9//acvHzFO/ew/jrIi8BFCfsDqfkS273y083QtWn5AhofQYaOli+2rD8B2o+U/bT2+RMZAr64PAOFTEt1/BG37+vrqtEGMpjlDjTqtmfOBcrxMbFevaAfLZNqCz0OHBlU+B9BN58oH7eW19l/PRs8+bX/1Odg/jLOztAbGLbz9zoPo/OXOF5ts5GUYBh46GyvbX70PLLSvf7lz/S/Rw4dQSEKzBXOoDijVflPlEIXax5FAnseJDpV8YPzn07DlO2R82ppgZdDbsN0y+5y2kJCGYKfgRz/MQC1nS6YttU+m47stpy7qRENOkrzEn4DogBhFJS9BIb4rx7G5qgL1gyppgDQt1LVvOlNUOa5OgEBPvnH01NGxU8DRAhvwB2Tr4Wb07k0AKhDIL8EXTpD20mZ0+VcohAHyy+EqFDxbav9FlIHC2vcvI1yMkJ2lFTLIUPAGQ/OhyvEjDNLJ4NDxI4x+Cb27NErqVi1UEnFxEE2tBcyNIyi3pzKhzUl9eAjM8BX2kXQ8oQ+WSV0/MAhVTfhAK6jXqcMbv8UqksbH9So0eE0/NFwmM9jREz093tXL9PXhM+0b0qYH5lUtgwCtUC8FHjpyqUz8lk31Ui08Dc9z4AYGuLs/r4/56POsAOr4a0IMSrj96hlbRu6YJXMzLpNJGprQ4rQ+qEnTmHTD0KYOQLJ+zLQRcZAVAxyUmk0xcGhRA2XG3xDIDUDwMMeH7fqmAdYwo4uZqmVhBDuf/rbz6e+3Hq90Ns91/vwbWB5ls+DuSI5WBw8RoXxYSpj20bQHtYM/FDYCS80vB4cGYxvKGlDeUtqX1rcvrqVmAkDm0P9jW6keGvyHsbzAWLafbu6cvywQowswqof2BBgA0TtLH0TnftN5+wPoNjYEf7aeXoK4pnPv+s4nq9v3z3DLCJqmbX87q2D6eyWraGEV1eG8VbDGb7EKySpG0ChA14lNsI45m+BdfSz7X7WJEckk/p/Bh2wju8AH4sXBERkuUvPhdgK7h+jWAzICeCGM5Z+JHHp2mcp3QJG/l71UB0f+YTA9DGZRRHU1iHBCAyJUsxkoTbdObRHHhW4Ica5OglZT8TSn1aS2woMZD4MZ1lRj/WhI/UBReeSE4anhvaifbc5Dn/HqhFaDOLOAzBySyLVlFQ3frEGVMudpM7AHgo9gGsLruopxF7y9Bp/wcZyTMSFen6XQHvZQCp9Rf8ziS3HQhyTVTNzHWpZFbyGoyZZl1w3Xg/2D9V/U57KCKNgH5VNrajoUimamgxoNdAX0y5R8UBVCZU3KEPzzxsDY+EQZ/udTAxk5IIeyLCksqBuygNJoFwJg3EZ4sON4q2XBtsNAC0jrmQW6uHtsUbkTqKVuNcmPIWyAIUtgQCUcEIdiBY7rNzMl+KlRpx7MWbDTKU1aZlBSWdNJw8z2nTRw7vmyKTNLsOROYTwaF2VZjqWjmZ4HgyqemlRT8INs46KWUxC2ewEKd6HELRtxiIu+JCsLiuXXxTKROsRsdPepaJVFrrJWQI0G/KnDYHybWGvVTc0KDHPWtGxz0oZtYNaysBGzIo3t4xXOLBqSjrbELYf9BTD0Ar1K+wEu2CB6MpwqjHKKhobtK0FIvTK3WgOfA9wHnzaYbZp+s+XxUv1gBdETIjzbBzyArSy31qA2DcRLNTdAyBYGxpuhFzJK4C5yx3gDhnTBkORRunZjCQHGJzhtFZ0VSpVqlj81oYoM4dauNBfUSylBpgBoa8LeV/Jp3hsGqGgjB1XZ0gV7Ur+s9WQ4TGo8353CiVsOIkRZ8N0vk5FmILOQbaP2kEQ/UZInLlTYO7JR+3YbPiOqXuNnGmVMTyh0Py49qIwCFhgg7mc5FABmWA/wwbMEc5iC6KvZZhCQU2wRspypk2ZthvrJzl9OjfGMWvvCSnT5GvmX1tQUND9m1iiBYLR98d7OzXMQkibZPRItn9u+fzF69CVPprWX16J7N3auPiPKkTIZ2w8201551P7sDgQynT9/3blzb/veg+jpVVJ3a4ERUOqQ9pX77dUz7eu3d85fjD68yOOSbMqA5TEMXNgNQwmo3YCYwgRXAzfiyBu/AWw2rClwRzOsTRsBID5GKm8ZsCCXySzImyfA9MFKxaigVwEHdX0IYotgxvIMZEqvyFAtUkkigZmUs5SOGDSIW9kurOOiMGmJ3AIG4+IXU9HQ/aHIxcyWUprywsGSmu1wpMw/MU1VMJds46woMkVcHkBkN3FlqaFEBJVEJhgX8OK4JMdBoky9VxMwbReDgySlpx2dpYAEuYkbb0EblgLU/hX/ornj1PURsOK45Q9IdHY5Wlttb3zUfvcp2Gl08zFaHYtlmYEut28tRSsXJZWTrYefkDgLvfXsfWgZrb0Dhrn9bG37zmp0+X77/hWwuhS0AmOWRVKyWpXdzS7wbIw9WRK1xJKA3N1YYJfOFLmabDW4RQr5YkBRSe0LFlvfncOFlvORxb+QnkYkhRZoS0oJ32G8kmRGUqCBtdn+hRGGxJkGPXBhjo1XY9ZKDdcHzZn+vIIkVbWwq1jT+caguwmb7n6dVPOcglkrgoZKDuuS6rp5nwS5zmQNB5tb9SBZ0EOWD40pjo+m5EDQ9XDeozpvaLvOlNpNS6glfsxZKbfijDmPsScFVhBQic6bzbk+wCzaCW26Ts4KZEoaW3kUYeMM7nhfpgIJkPYOPNRza9PAYWpUDmUBDwcYtq5ziMGVPd2+tBrMGlM/QN8GY4GNQdq928nnpi2bMnNL3R0DKnQaNRf7BXmv2huM7dm/hEUJCYAIui2I8VAP2ESUrNL356aYpdsdyu5OToWNTqvRgICSLTUJxMIwjL8ySrxBfQbwepUtS9kRM1jQPTSGSy+SeDae6araI5y8EFJ6woowrG8DK3HXXWCl20q78CW1UISZGGLQIXrMYrrlzOSxhOEI9ilCEG3WonNKbvHOeFZuRriyA0HGw0Rho2/gUhlt+vO9K6WFVvNaoaKwmZaz8lP5cZvbCvGoSN2VWBaK8//o6Rr1QrGgH2vZ9u6cFRoPd2Om0gQdHTAUgwVGcThIZ60alaRyGjaA9SQmETNG61Z7hy68fT76Pz0/Plomo/3VCS10FT4Q5gIcY9J2azMJ/JRFy+roCxqm06jZbkDzEC8pm2k6id5fdWumfQxsIhe98xwfHpff+gz+bm2KWDz63TpG3VfW8XBOygNevNP5eBOCfZ4Q3Hp4iQfm3WlBzD1DWH5rafvMlc7TXw+k5/gvCM49dpr7oiD8wKDxo+EREVSKsoLg22k1vXk8uXW8nIwcDCilUt+cYwUarpJ4rKtwRrizQnkL9p4j3yXUzkcZOJDBGFRg8EyUoZkBDqvAsDDq8JDaO8ioZGv4GTyufQjKQFYKhSayTT03EDtqSYjkh/EeE8EuR1KFnWM6TR4FsIkysFJ38Y7cKOo38cY9hR7xSoaT2s+7HC6cQhZEJEGkgswCuKSmHK3xpPdofvBEg7GxxXp8MdSnlPgi00uo+WqBOIz5vYJO0vi74o4HGANY8xNxb8JlWUmAH+xAwQRbZkjj9Knwgx4+HrNgNiGCQhHyvBZrSwO9OhznqliqFGkLVOY5EWAOKKPJVTCPwpUKsZ5VpxkfzRtvko2Vju154kyJNxfAMzoAJ5bhCojxUrUcP8iLidgKQP9xa9TCGJG1meC2UCAFxlGqj9Q2MBcrzBLko5mtEGA9gKgLU5ClGKwk0VEH85F1XVBi+TXWVs05gu1OWSFIzohz8Eq6dKm5lgG6yzGt5sOTAftw3wXf4AS0BsTmsO7ghZhpE+y/H5GEV+EU4V3NLHZpOwk+UlWilXOS+KZZIW0qUkMnjda4PTDnFAbRpKZjCG4likkSzcmmyuLW/D4RhJuegtqXygfBpFhCrNDMp6hDfTBzAwPR2Nb5fSZuLA6di1eqKh5C4akT9mj5FEKlEczqecaMfrASO4EuVMU1VZRO2mtuiC87RVEzZ5Al9sd5fMxXiTkzMJgw8RqaLksXCkTKks4aAbuI1X1uo/EKQxynFDrsrl2AJOIO5mval9Y7a+fatz6InixFa/fJ68qMSjrrS51rl9srf9y+s8qx6XQ+5B6HWRdE24l0ZedKXCYhMsnMrjqMK0uX9xDTqeeS/3jFzwhaHiqJgk2obLcXryGcVt/358Ox/wYhmGDqw3x2eL8Pds0WjIIglmCdkWJd1kDl1ReWZ06cbbzxYRwsFP16lN0um4h9XniWZNbg4rQ/m51nNg5Lc25bPcuRR+jP9WaUdFyOwqxjWWInxZMsPmUYlpo3zWCG1vHGni0RJz8ms+OAcWUBM6V+y2mUJJLgIJMc7wK3EeIEZc7qVlPvlxZv53R6SoFTabZs2JS4TQusnlEqI7LnYQz0E+9RgUBGXoygTrhrZmWW3T19/0tD1rSyrjYOfE7swdkEFRFOvAgBJIiJj4lkXErZ7AZ/geoIeXWKkKeMh8zm2RVMdAg0RtA+t+H4cC1gZ/ozXiiifwHjLmJ4rYFZJHHshl7C4jA8vgtCTLOW03OJZCuCVOLbovisJqXxRdHmTN3yFf4SiEN9ehpc1XBnpNRU2MT9CuuImgakaTSs00pJg4pSHACh1HEGykIimRKbQWlUiIhJ32D3M9T0skAJ5gdN4G+vBjB3aIASSMtQFFDIJJKWSqKBSllQaZtETnhGGz9L9bFEoToRLqtdLKMgBKgFEDx4tlmjCpTxfWOsxlqzzq1EMf2pIDmI53YeYzhDox7ntxy8SzWvVYotL+8mo1K4whYM3Fg79YCTgxgCfF8zbdsFUG0cGBQL2i6dHKdHczwCZjfI9e6VSN3DUrSHFS4ZoseiFdRMm/pJV0SY4xD6nGLFCcb0hBTGJCaj4nFUgQBgUZg7QXMT12PHUWMav58+kZwOs0KIt9A1U8FD7/FScgOmNIEwjddmMr34XZHsksM68grWS2qYAhOUi8vWyksvQY+uiJwHlV58aYRdgSm4WyPaMkfwMFmLR7tiTBb8w/aBvYzFD2xKZq3Wakrj8NPemFf2kkyRvcEMufpEfMuqxK5+YCDPgVAAICF4ux9jFOsDZapcuVeMAvxgicnCSzOMctfNGVY6VxfMsIMMowBiU5BIAAF3eGXxn1i62crNCPGvFBgCfeM5DpBS8l2DUsZGwNpaTcqsVOqqsTlmrt/U0rwAHkIoUmsM8j0IDGp4V8LR430sn25guI49zy91qblli51mSLhbmxkXqC3tvBCbixoickvNUgmKMATbMKCekLdXiWilVjJwS43zWZzaDM/zpxie0RVa2kilIgU3ksK4nmIKCcpnFZiJvXDERmn8DaafCbK1eYksSDJfJO0zd6MPLz5/cgMnqS+k81+M2dIXJPYGqnR4VBtsLJ6QV+uF5HExNTB9IX4a1YYaiyVho1Nei50yZW//4M6La5xVKhW1aMkQq8pPT/5bqS8zvxOo7gnCUU9fkCBwkfyC+z9Og8FNOgWo4sgDPQQCZSpF6LUQs4y0eIC2kKwGi8JjElbGcNmc4JkSwQl7XoQN1pgoGMMXhk+iIAWsRfx6Qw5qRsuLqIsBZpa/kOEsnhS/RATNFLngpRwdmNwhNrkjSE6NNRI7Mbqezb6RI+elMNsNPOXy30quR1mCY/E8lk0uJyZSYJ6vmKE5QXianKfHeSqdLOSGWSTRg887m588f7KaXDgAg6aOwhmNc8SqEBlCN8ubyiaqCpVlTxPl+xapz7L7UAkkJCtEpVtAues9XEDiDFVIJHcxBILp11ynl+jwwJIvTtnLH3rB2WhOkAvyaGgTBaMvqt9GlAtSHbKwyETFuQPwiDlbTGwrTG6e4aGWhn/idCPuAQvKMVWErsAzkAWpqb6uczbcljD9JLvx7EKYOER+Y4Z6630dMF1x5fuAfMVlJfEtQK4mfOw+PPam2H7JC/mFXYNfXew+ifMgoLJ9FkzZfl+mGjtDICCSZajC0DUcsJyCk3fkMc7ZxeKT2ZEyFjngKTjUPF0m82zTxHQuHTTk06rf/x56lzSr2qPhd8+yzhdnVwuGEYnRouAzf12ARfy4yxBz73WaLBqyD5a8VdlmByyvrnTzUnwzIuZwt56pxRSnivu+zQREo5bDHgwFTLh7ZCYovDedRDbJt0ZboWXDns4GPElaGUr3/Xn2VcQC1cTCg7kVjx0z6NXxYGUvt0y+R2553gBYzUlbDgv36/kNSNehAkNNaJdotEt30mXiH8Zr/pRBZ6k/z67qkBhIdV1a9gpu3IT8jrNSpf3DZRnCST8H9u5JOskGrJrcMs4Mw1fWLFfdt2XcGQNpKE5ygTm/LwM/rIddHc3ZKUNyViargZip7glSj51dU7MuYY2caYNdRFDgTU0KED+ZjW0xNQzlrj9vYH6iZrKkAowO4VjvKDeP3uk6X+jmDbZLIQv4d3919PBwfXFAjgV5HDxDuoL5wz/SqjzcLfUgzMS2EAuQBfNE8TybLGQOeuIGeM4D8I1U1d2o+hBt+BCOsrB6yiELie+wGBUKF5i2R8tapSEC4F3ooa7IAv6Nu8+C5wGPTCOMneNHujt3q7AgAMlYYEHc8mI/w2ObnKNJjXRdjjy6/W02OZjFJxS93nUqnI3aioPKrrPhooXE9AJaN3Dfr+QdO7nGf2C40ss6G6XDhw+T8aO4Y/0JkY1ycSKZiL4QP3Fz+kU8MV4BD7ESBT/6QsIYq5nufT2u514mg//8pxX04lPH0tg0BZcPQnFoaDohqVtBzWUKxJ8gqFkUf2HBCkpdZ5MjlcITsl2ldYqxA5E75+uf/MXi+cVSIz9OttnF85OSBnGfHnKI8/VSkmdSpHh6pO/RZveQwhcGWbDvy4SLrgdbDmlwWCx9qxZo+NMUmNoo8cQoP5TVS62w0T+CWduANIqn3tDmfAheFCSg1VtNL5DS+fl/chKez0vkcXK592zOPT+1cm/6sfShF/9hhNSXhyAgKAlzz1TDO4Ao1iZGn9QnJWVyQC0eF2B3Pyn9p1PaCzAxA/jWwJSaTzaR972aTbLbPQWjT5Doqz9vPXs/2vgtv0NH/vvcb3I5LkXKVOXJJ9mAhuXA6jyfT31o/LqOfDDCffTvdTKSz4gyraBIv2E+tCDlPjPOzp0muhLve8ybiqvJhRApUmyFtzNYHZbx9+xNDZ5Vp00vvq8h0uzsdLoAN7m22eVkoYMmywOI80Ge0GdHnNrL/lSrCTu9k/jmi+gsaGE4BpvROky2NclaQjBeB/PQS6BO/JYy/ypoXb6pFfrs+7iTrB/vpIi770I4PqsyxZhKKf5RoBJeC3OB/0BnNwbEuYyKs2uYLRuGbe5KBsAwz1SZTFPb00vbX78XnX+89fCSuC9681709OrzJ8v8p6m2Hr7bXj2PP1V13HWnYHljPxT0/MlK77FEUqgkMSd97ezYmwP4jfQ5OtlP660XUhGpJZmY9DtQvbuzKAXT4JgdAG2n/avDPTuNFXeoDA717JLuzAv7DsVSbi9tiptuIu8lLr7BZgOPmCQyPYcSi0jRMIP4LYP4/3hI0GT79nn+A2QcI/F3BfhP/fC8U4W0r3+JvxJ0dXlXjbLWhQMng21f+IilMqPl3+HPZ218BONEZ261N+72Jmv7MU22+U2pDtP+3hKfq/foVdGqvTux5FrhFA7G388omjgm4CTnG4+/nFvmX4mdkMwS33tPNd59FquvNwdpkP/NmU+W4W/eldJ6sT0P9uzDz+tK/OQVFhcIg1wfAB6gprdcxPFxca/YtqKzX+xc38j9KhvJ/CrbLrbLT5R3NV7+KxzRw4c7f9qMzn6+9fhT/Em/969zX0l+xa+98Qdow77MutpePdPZ3Ig2Pt56dBucKDq32dl8b1cnSrcPMpglk8ydVMDr9lf3o6/fSW78s7v+vFVy0b/oiv9KHKMFXsFyw8FTcBl4OS6ZLnrUxT8EJ3Of/0m43n3TpbuHB1R6dsWFvafLj/Tuhst/D8sXEsIwgS3k4o4TvCrZ6w2wmLMkCl+lpTPn7GWWgrOffFQHgYaF36vAQz/DYEQNg/3MkyHo8hik738A1dYLQg==", "tools/eval_rmp_138m.py": "eNrdPGtzG1WW3/0r7ppi1AKpbclOJqtFqSEFgdmahNTg3XwQqq621JIbtbp7ultWNBpvGQhxXiThEQIkAcIQknkkToYBnNiB/7Ljbtmf8hf2nPvop2yHgtna2hTY3bfPPffcc8/73uvJycnSzIEjZHTur6O/nCO/PXKM+Fev+pdOb64tb62eDT68s7V6Mvjm3NbJT4Jrf3288clESSajv3y6+fDM5tqFrb9/tvXokX99ZevGjccbp/1LJ4Ovv2CNo1Pf/GP5jeDOenD7hn/9yujCCrz61+7515c3H33sb3zlnz4VXP6j//AraPcsu1iawp9lElx/a3vlne0PLsGnxxtnJsoy8b/8ZPuLU/5nn0Ef/7ubwbfnRxfuwXjHykDQLNlc+7O/8Wbw6cmtc/eDD1cYKNGavSnXs5zBVMNqalONBdUjo09OBtdvBg8vbS+fCc79KVi+RVynobiaQarkyK+nXsYBZ2RSAjK+/iK48u32lb/79x4CJf79N0cPb+GY+2DM7esfB8tvTPlnb8Ov4NqZ4PJpDry2Fly+Rzy1R5A9362OPri//af10eWPty//AGhIs68ZhmJoJo40K5Pg7JXtN674P3zkXzoffPDt6OpZf/XB6G060gyM9MorL5Dgw3ub6+v+2Rv+tdujhz/4d77afPD5443zABmsfkAO9wyDHJ4jiy55QTNdjfzG+u3z+AaL+eyrRw6T0e3l0YcXgwt/Gn3+Bg47MfrgdvD1ZVxMQuyBt2CZxLMsw53SFlVDcbq2AjLRle0BAd6ZnmZ6U01HX9SmjgxeoL8FyJSheprrybZHikWr5+0JTweY1c2S/LprmROTk5MTete2HI+oTttWHVcrkIZlDwoEvxdIV/UWCsQduAXi6V1touVYXWJDo6HPE97zGLwKLGavC1SrLjFt0QQi0FhIvMimKbd6ZsPTLVM1EPrwxMRvX3llDoQAkUmK0tINTVHysqO5lrGoSXkZiIN58V8Teou4niNhpzwxLY/oJlIpI2kV4CoJ32QdlsTxpOlC1CM/wScCTBezOK4uakc1b2LixVfmlF+/AKTsmy7v2z8xMdHUWkR5WbLzDLENn+yaTQ6S6TptcDSv55ikZViqJxUlmzwDs5cNqw1d8rLb60o4IEPT1aXXLd30OC4XcNF3BkbbcGrkuSqZZiCxEabladr0uuhFpogbJ6GrnpAAqIDkvk5RlvJ58mz0Og2vRfbqwHwNoEyQBtQ3labVVXVT8awOCLKkmY0CEW/V2WllenqaEw6CE3zzhr9x0T/7ORgbMktG99ezRoDZpBAHCD8IPRgGJBxNA/xG44C/wDz8Y/lNFEjET5enqXqqq3muWCJGImucoGCUpw3LMLSGJzVRSLUTntICwXU7ul0V1OK/+V4L2Farhw0tyyF6gThWH4VHA8nVHNAmQBPrxRdEJ89RjMkP+A/1TTd7WuKDByNxSiTAn0+jQ4H1CIwPdkjy8oC8PP2EqGEaMmDWzCYujwz/A/8Uy2nqpuoMAFk+A67aNoIzuc7QgiQAVJ4crIbrlKVl3tHUTloemTIDLa7lII5aRSCoF0jTG9halYEYltnO8xWjEoZyP1xi6uSAIEu52guwrnUwnSrMpA3yJAAXdZW8fBh1V1O78EmW5RybxFMEXCEKk3RYN7Xj2nzxRXhZ0IwmmkIG03QVhKgmZEfKvdxrtwHXYbWhHT4+1YLufegOkLkCyblq1za0Ymn60By8urahe9Wc5wA1uUJER3XO6Wn5+JxqOURQh8FiIonDF4ihduebKnEqxJHbSABKB46Vy3NRLU1zBj1FwOEy5ZDmdHPwKjzqmhtOh31KT8ixTM2EmavmVKxTRD6YfR2AweDuNQeKPzML2rrrPELywX1TnZaOMc8GWD2nR429y/WdDek5g0jQYAzaKz0xXe16qtMzp5ifpEBKHKVSOtBRVMNWG+qTrVZitoguM1lsjM+1NfkUGfIJx4bOFWDaS6+Z4hMInd3zROskG0070dDAO79If0GnvWfcMgae1ljgEy4ijFss7+v8X5sdX26IoWhoJ73atYw51egQbVFzBk11gAYMHl2VLX3xIMFgqqmr4Bp7O8kAotpFW+cOTbkwjgfjoNiJoYqJoX4Ep9B/tLoeHVdyUoa/67bRUHEGdDXXVdtUqWr1pBnl9jD3mpmT0TFLrclh7j8g7Mihhe2y/o5laLl8tZrr0Q+aAZFi7nnX1V1PNb3cUoUMOSSP4RivJ6mj6qKPQnrGrTLQnl1laCxEc3sSYRzH+qaqGwOFrdmPsCZPSl8khRHvEpxNMFJ/ulytTo/lXI/zCRx6L+nO+eqFU4C1ywtjhT06BbKIPTi5su5pXVeKSQJzUK0cgchp2Knsc3E89JqL+UphiTCHx10SFwSOi0dWXbWjKfOq11jQXAnAIUw5VCBzIgT0HJ1agUMQN0pzEK2VGC4TFwPjA+iRJ1NTHJJ+U12MaQEEnHapQFq5oxBRaKbVay9wgog0DDsv5elMD1WHh5Zg4OpwbinH0DgORioAA64bhmcj1OVFXetLJiMzIohPriZBtxowugL/FUvg6qP3UqXOxtKRpY5qtjXJzNeBEb/ikb+ltB21CaEucoZmI4blulIXjJJRIJxLED1oi3oDzBS4YoVGEpxbFE7Gfjxa9iwQI2QVjXunw2U9USADpIFjjJaTfqiSE7JnSWwUcMGD2FsI2de9BR7jABmy2vOshuqifvaa6GtYhBNSWIAFUOcNrVnlmEi1ShhsPmVYjDaoJhBBZyOdSBoT5Ad8Oyw3HHhSwBA4kJBJRltmGQZNihZUW5OKsPTQ7Oq/x+c8nUf0LRYJApPIs1WeoSB+KuVSDMLE76WJRHCH6QVmFDCKmd9pCTH9BL3u2pqno0FQVLOpuLbWAH3Tf09thFhcrhVjFheFcl+pDAJXnY0nGOe2Vi+TEiQRrOAB6cOzhLeWMbWg1Yh46wy28krBmVg6kRGbvi3YLxvqADxHrVTn4ai2qNgGrLPeAJC+LfMX3k2Ovh1WwRDR5pcK5CgDhif4eZQpcB/ajqLmviRCLMVUwY2gYoPtkoTJ6WgDVyyGiTkY1/ywQ+hsIeBlpQ//4pvB5Xuj9c+27t6AlIrVdfyVzzcfXQ7rOuQXHDhdGmG632w6jGZ8CmWrYfc4i+YVlX1vWI4mw1sSBsmRjsJiMmgVQkSV8YWJiau3u5belKAj11S1F8GU5GmQLgl/FcO+gBkcgi2BM6mWtOJ+Pm2kD0TMVWzNUfqQtkZ5HOLsas0xX9AG9CMj9FJMA10sNrkGiKDUx1y9XyDwgGaOvkVKofapHpqW00Vh1iSkpOYamN3o3WqxFIG6Oi4adPgV/JA9KfpitVroP9s4pt6t/RfjjjYAw9hPZknzlmXko/w0M22RyDlWD36yxRDY5a6mmhKagNm43qfYMw5DtCw4M8AD6Bimcj6ekvhn3vEvvbu9srJ99RTk9Cy/ZwU21MO1t7C8eHoleOeL0YUVUMN41Y0qIxYZS4TVFTlaWsSoARtAEyGopzRSO6HozROME6zOUcWKyu81MIiSBCqGnKOKkhcchM90OvtnhVuwS0pfcXvdeN+XdgJ3wdY3FpRGcqiXdsbeeWJYnA2aXC2W8+J0W472O2V+wNV92KxEuI6OwUUFuskDFWYVlqIgpgnut4mNqVJGaEBidRDmENH8xeMSEa1RLPVapbR/5sBsHS0yBgHVuVhcxx2UqEbt6XF/nNflZrah2uCGtGTJ5p/rlff0zNHsQ2f6M3jpjC+mtR/VdgtJPkQv4G0g3U/GyWjuIBZHP9vHwB9XApFkp/jE5i/kuKegIex7NVmWYQywEglPINOiL/cIKC2FsWiANooHfu+BKNM1tBMVtLx1ZD8SRUua6gndrWJ1t5TPdkwpmpBuGBlxUIr2RkJtCRgkwkdVnTaGRbQPLGA47R16QtAhesa7jQemJqvWr0cSJnCIQnKGraHpSnSTBNU1FpaTf6mSeBNG7vmdcHL7lqUDOlNBTmR53Lxx5mLCTN0L05QwhKR1RmYamJua2H2RyFSVdcW4fjxIWFsPC0/B/ZVgeT1YvYi7R+dXgs9XWNgGfoU6OTeyJ3zfCbSMZrygwBiZit0hBljg/+8VUYQFeeQYikPkoKI0UoFJIByvy3POsFdgCAQ8/xpBa9S74K4DdmPA3gLdVIjrVlNTESdfcjbEc2CZS9AB4ERUECOCuRuxG7A7EdHWHN23ENCUKuZ8k/BURMCwUXOVFGfWM5KrAkaAsZH6CjostA4JcU53YylykVD2zaWQ0MULY0vsyNFSOvdl5uWKcIi/5/8tEgXxCZ6hORIL0U5bYqliKGACYJhQqhwC5CpgopPNDe+E0tQa6gC/gX0Xry6yKAkqSANIpl/wXCAz+RQYSkQIgi9jYERkCXCZIDM7MFvIaNxolceg5qwMgfk76nsaLYtNATIdpWZJYPwHMYhmhi0YokaQS9wQJHK0eDoXz26j5cnxPS9RioAhIpsWYc+hKimZRWCaB3Ea1bRQfJiFS/eO8X1c78xajMeSZnESiZDtbF/1xC5dUVl26sm472AuD9Kp2cqii+tvGFZ/ByIifakVZ7AkxNRxzOfKTD3P1TOfXE4qID9m0IwYjRs6A7QLAfgd1ypS7jg7kx4JwIadClmEiJam9vlEbTEFLGqMSwzfktgcdnqmYpmGbmpKo+csauAvXRgZVLjRalMnCCpHS9pKWCgztT4V3bDBMpq0oak3vELGs4ehmNbwgA2Acnw9xnCwsKvZuO1bYHU6Wmuv7ovqM5vrF/xL5zfXlv2Hl/2Ht/yVh/7Fd/FYxLXbJHFsYnPtbOrkxNbde/6jy+AngfomBM6MbEd3tTwJD2IEV/8GfdJVHFBqvlUvPfMM8CWfTiJ4VQdVERdNo5wQrNyj+pOt76A73z71XnDlW//U+mj9fZbxipBj6+73/t1P/IurW3fvjNbfYlWZzbULBI99sAMfYexgc0mAQW3VgfjFg4EThWZIC37X0yE3oAU2RaIEcKvmdltKV3U7WFISwT/uLdAdJMhoWj1QqRYYpkROtueYY8dN1vFDPDVRCzIsR1XUAkm8zxeSHJ2uh3Bj2+fre1DBZ0+jISM+1SYeslEQSWy2Sdp2n1GS7p1hk8PSFTDs2JhPkc31t4OrfwdBpzJ5DI+T8ONWn5z07y9vv3WbHPnNMSIdhwB4Hv4/Xs6D9G+ufRB8seo//Gpz7Wbw+UZw/kxSI+OSE6f3eOnJ5jX/hHDHy7vDxWSOJdysPIEFTmE0qun0XfSpJWZUZ2HZDlz9X1jKf+Z0XC12UkcFG0b+UzV62ouOYzm09s1VmGogTSzs3ZQT1TqlDMw4WbYXkg7Peld+vql2j0sML9rsKprtvqa3FzwWR1an5ehcgX/6un/rnP/ZQ7CqdJeBlXnRMIIdBtSZ7ZiMc8luy0Q4wIJjIQu8YAbP4rieCQ8Z91qhe6S4qSdEpg1z6JDA0YJDyiFlPFpjsYCYBo1d2Hd0LAgOCWTKD+aEt+FuG0AYNax+xSazBGj6luN6Sgya7jKxOCzZulSfiHkf6qR5IQMRI+VR6ljivpUWXTI7UwkHX5NoV5rxPM22A+OfY+XiJ6+wgexQ0WfbOK7mQWStmOBUUhu4/3/2v+iW17za6PRVpxmrL+FRO2EUdJcey0LnWsmU1ujiMT3LFtSoviIv4ziIajZ5s0wpAseHDEFrUxkbllEoFHQEqYNsQFCC+S5HYjb1LpWDsZ05ULdnKJKYEtvGfYbiTWwG8iOYPU+HHLhh6DYzl7jPoYTWJJFao8igIErcmAjmoWw+HQsPcYapk2ygvz/VvMRR/ZOsTGi/dV6o7wjDwgetderAfWEb4A0NzRijkSY42nQZZhZOGDT8lQ3VszaOszKbTIfg3OQlTSLQvXvPmCWj8x8DkjGEbAhMrWgXeRFdniulk8dk6prpn0qfMXNKoRuHcCkpFkmTy8+FGoRLBYiuCJbTZqmS0glslLWu7Q2UhgoSKSXOPdDF5KmaboKvthXVoxyX6CdIzVSnrbGmKEsCZxss3wouvucvb2zdvTG6e8V/9D4Ri0WCN+75Nx9tff+9f/peHAGBjAOCSJJkPGazcUbGT8wmT13wiiulLG5+m+C/miWQcPqlpqNJqdci4QG55l8SrXGth+4Hqwla0dZhK2CvZA7CSnHIIoBgTi4BOH1O7u1k6ErJHZpFD41iRGMGorgHipTwZIZMSuqYEdMAxd0RjD0qxsSfRytcl8KiOIdBN8K3esRBIpRwxmEVU1hxZl9+3mn3uuApj+Gbw+VWpVv7isq/gd/u2HhYc0Ez7GqO3joJ/rgcfHZz9OYDSGv9708+3jgdXSiYIvPs6fHGmdwOGOmlAwwFtJbaM7xqLkS/9f37/spD8u+vvnKUMOQ740CHhcf5aDyhY1lT4CsLdDSfpmX97eUzm2t3ML+6uOrfeJelW5B145bz+kdbd78k5akDwJvyvqd3JZxGYGMHxYoHGzVetQju3IRRBD6HngwEtJT/iNgVtkfcdNA79CQWSxJMejCCN+GJNIWe3NbNtpRr216Z4xUWihso1LaYadJdRV1UdQOjLMgT2Dm4ht3jh7mEzwuThHlqXkv7EU0mJqNKm0Q+3yrtV9yejfRrECfxIRgQx8Xm2IiSKCy0SDh/GcULb4zY4LUa9ORPlVInUhIXi1uDaiyvb7TwWAStzzQ6tRy8CnWhkQVoZoed3qOOskDEthk74qa4mmbGgGKtMVhxfK+GR8vrZBiSukT+QIepDvEnvvGLDsMYoqmStr8il1pLR+A7zxGH7PdSLnOk/UnvUEz85HIWsot+EQzzpqmAdTUZf3ALwM+d/CxntOLczL1mVlP/SHgtjfyCHhehZ7PgeQae6Yksku6SSyzQpP/ga7zIJi6vEalU9e+eH31/gV1vy0OqhvOppSrxdVw51jd+u41IW9+eDS7c3Prh3Wl4YoeiUjjCenx9aTJJyyw73rLDXbUKwcGWN6pxZKK2TukJri37X94CaxWCxKrvmdEYf/AqmRR8d38GuoHxDb55G5+AZHaIy1N7ACHw7Vwcry+dAAKyN99inXcr6EP3NHXisE+schDyMb15kppbbnv5Em7FfnRhtH6qklrwYe443RU7OLtEhjm+DXZwP77QLRb2yDa0Du7DZ5Vunhz8JT7jNpSACfc1Dh6gn7DIL+DoZOnLZJSP9116koZOgZX565kDv5PDvsu+wseDs01Ahg3Rlh207pdnWrw9EshEMyUfmvbJZWiaTAXGfJBIEIFOeVb0FTtvSYyhGMFsY7B8O41iKIejhxtnoj06uA9ijR6a3cu8uuavPqCXabbeOh98vAoCPDrzwN94E29sUr9LM+UO3vVkLnf3QjqeiCywE5DRqUj2Ew9EYhtbDHHsHHIA1XHUgcT1JbW9UhfHyPkaKvPtJ+pEb+QIY9jCG2B8xCLFMZHZDfl5TxDCClJnaAhi2y74VqmItOAplHytQv0R5W0UZCcoEte9+FAYjwqs+XC7KZ9Me35CxsOlgx+oneVXYEE6tlavQxSMdwlIIi4KL9CKe7PBlVvi9t+Yu354ATbul8e4kl2u5RIJqKmyywMQE4IM02PuCYbll6aGR5eYlOZ38jpzEKEXyCH4iTc98ezxLAtJALVCc8mdTshxQaxVyrP7frmfHZGjyOaqiDQfoWEnf3ZFQnFUdkDCyxW0HhHtPKJIV3bASsW9VimVywcOZJHGti/ZLa+d0PA7YE+CiF4x2gkPu3+0Bxp+jLGreQtc/0LsUritRbdo43UAKb4NVCBlrTiT+BruK8DCjvuGOzkFMhMhrUfV5thJTXr3Bg8bYXmd1usZmZEiYWCf3bmNxWdi95ZhieSrEApJuGnr7rRh+2M2bqvR3i2zLfiY2sOdSFSo3FpXHN1qROdEMWikOw5Y5IEgtqMNqrH7f/HqQD7jOwmpDSnSSmnaXaojDkXAV4ccdxxFhbqyPxDgCMBWUnWPWJd0Ol95FnsWknWRGHwqG2fgNJj3IJWKAUYwoZMMo91isUjilgiCqa0bt9HWiT3ucDFpHARu8tz2x++gW7103n/wNsZfV759vHE+tI2Eg53131lhAGAtIVDdfHQBA1dh/sBawtiZwOkIlcJc5blSGWOcX1G2TstlcEPInimcdB4+l2fjn2d3/7x/zGceM7m9bld1BoqHqecY7aDbw10scUeX1ZiixOIqx+orMdViV5wNN3sr22vT+/wSTKhAgGz8sT+1hYCHxqHnuEKcEOcCIkpVX5CEGv4dAPyEAj9kAkXPZrEqJVqzVBnXis6KtXLDPpehKTLsskeaqPcT5X+WnAfX/sxKe7nYlmKcl5HmUdKyMShTIrbOSAfwmK8bfSsl3sp1sWi8jpRyHbQAVKF/ByFMg3G3LF6HJok8uhLPtWNmXzNU2wU7FBWnY7knBFcePUsbP0QEEZo4JJO8d4e3F3gtHtnYwZOqmVBuKe67wHwnDCGSnjCMEaxqeroCw7U1z9PNNrPO2IE97AJJlwcxx5dLeCweXlEmWj0vEk365ytEc/hHK7qdpu5I7MWlu2xgj0/oaMc6qU23FIK+A6xR8Ja1hH+QQ272urYrObhZAqvSczRFdRu6zooqBeAneEWvWs7jZ1ZfquZ6Xqt4IJex0WDWai+AqNbJaP394NNr/nd/2/zhun/nI/Lfp97jdRKggZY58O9tKPQqgaLQaFJRaN6n8IiSVSYn/gcysn9/", "tools/eval_rmp_bpe_multistage.py": "eNq9GmlzG0X2u35F71BszQR5dOTYoCqlFm+AWop1qMVV+aBSjcealjxoNDPM4QPhLSckcQiJw5F7k7VZEghhE0NthTixIf9l8Uj2p/yFfa97LklWDkigqGimu9/Z7x4LgjD6zuskeHA+ePBNcP3K9qUfO3fudk4f6a7fDm5/vXl/ZWv1WOfuJ1vHrnSu/ufRxsnthaPbx8+QCc9RdVNxWrZitGR7boJsAvzig+6VY2TCUD3qerLtTZAcmZgMnx9tfPzLwtFMZnPt1OaDz4KPz3TWjgdnzpOX9j7auJIpyGRr8VZw6ubm+lLnq4XO8o3u0ftbq+vB2QtkolZvAIG1JTJxWJ2mYxQQd659tL14BhgKjm8Ed+6TiRbww+jhg1LXTcre3JZqGMlTuMFwuVOqQzVygOQnSLC2tv3tenD8h80H33VufhlcuxicWO+ufwFMP9r4Z6Yok87dI8HG2eDUytZPP5E9gOFW9+pycOdfW5/80LmwuP3vE8HyMrAzQTWfk/MsB9Xy09XOxW/Y3umJ2pTqsSWuZlIghw4dZBuWRtMbRb4BtE93Tl7YXriy9XCRTIyMGFZNNRS3ZTXheHD1JtxJcPX77tfr3QcPg4ffbS8sgxJBfcjzbpkEn57eXFvgl7u5dqOzshGs3u+sniMTdR91AcxMu6iZutIy7AlkH5+bE2R7/dLWneug/K17q91zP3CVdBbWO+e/xysuFOW9L/9v4Yvi3pdDLk/dCD471Vk9GxrQhe9BZGJYrpuzbYNsXz4T3D/+y8IRrovO1YXg3p1ccG0xWNjYPnIxeHgJ9mKlgGCxHoJPl4BgsLiw+fBa9/zl7tIi3N3WvfAA54u8pdZqqqMRsIjg7Aqzsu65m53/ngeJ/mIZ6iR5rZDPM06vZAix57wpyySeZRlujk6DRtGOJ22qtHzD011PbVCwaZKrWaZHTS+nOfo0zf1t7iD7xcOxkeViYycjI5bvEcc3Q6R9CN9zLTMjCEKm7lgtoih13/MdqihEb9mW4xHVNC1P9XTLdDOZaM1p2Krj0ui9Ztlz0TPDFz63VG8qenbnXE7ChkVDn4zwv4Nn2IY3Z+tmI1o/qNe8LHkbuMyScd82aEwdLLg21fMim6Zc980acqkaRHXJG5nM3w8dGidlhl8EsXQDhJJkh7qWMU1FSQYJQInhT0avE9dzRASSCEhMdBNZlpHbElwOid9k3XSp44n5bAIhZbgIlmmA8iMR4DrcJtUU16N2KDvcbbQbxoxM5vVD48pfDwKre/PFvfsymcyfQ6EspeGomihlNFon/O5UrzZFXbEFjmmUIhRZEq6XmL4qTF0VjmScmq7lZEn6rVrNEo1O6zVaQhGyRAXT0UD9VCIjB5jmK2y9bliqV+XiM5IycgEM4YJnAWET2M7LoIo8W6tbDpnNkjnUXsQT28D/2EaZzMqeJXLyEqyk3uKTM7o3FXIMrMmq70GAcT1RqPmaKgDzyGs55jpLqKlOGlQrh5hIuUz4WUlK6ON/RiNLFGCCSSPOSr2bEBdg7w255sCTAmbhgGGLRkNmihCZ9UypNhVHCllAJbv6B/gsMTmSPSlBC0oir5S5IkXEL+sebYmpEybuF7iKqWoqIRMImAMHmhWBlMmPOxRc0yTtGFbAw0KJOJZvamIMniV7pGxyCCJdcgYNmM7aYks30wBFuEQUo5gGnLS9QeSMK0BiWA0RTsek5sFw0U5bapPGdupZTTDKXkscLYF1oFOzX2ZxT2m1oR96jq7BFZNRsouI4+QVUuDqQVs0qMmISiSXC0+yPdVFn4UjB0DZYNjCGOiXmpbfQENrAgEitmPgeYlZ8mi5PToPjJbb4/MCR+M47G6abqUEyHaFJKrytE5nRBOEg+MJR+GFVUSAq+hZUoL/Rwrgfcl7oVTlxHT0GUc1G1Q0peqwKAB5Vp90ILYrkw2M5IORoKE812CQPsr1z2JY6EGyoc5Rx60UqmyrZnpMPQjyAQUnEvGwPBZRKKcdfSCgsLM11cb8A2gqHKfnzCUuHMYXhcWXRNIeL06jkWsGVR2x181fXHSJxRLTQa6XOoqgQOqgRhZ/mShplgcxolYhRgCE7PotERNPFGIg1UP1mFJQnw7HLJOmLRFR5fBfhkkC9YCYGAvKBfD/0IOhVGBFAqsPxOSuEiuL17jJhPeEtccz2x6DhausO/T9vkjBttKmyV0wvhf2ajglHl0xf9KRPXyVlYssvsByMbVIbTc5X6AjxWzmV5i5xZTLdQ3PifXaBhiSXsMkohou33iJdC4tdb+8wyt3kpT3O1T3UAhCuPVgv7P0bXflSHD92+6xHztLn29fvk50VwmBy1HGCA+XCXc0FSqgaYq4JfLHcK0HCu8Y4aDaQcGYJYc1t9DrZTbaJRceKiS1RT1QgNhn7zYkvfd9HTIfi1GKOO74qTRu2Uk4gGe9Jb+mqa3D4iBayKZO2XASUKj7FXbATVsxBZ0+DzbZ5QwhVsEblQ9DkmAPk9HD4WJ1B8ox5PNRTIwuVgiD8V2oImuQhz336eNruhbrdc5fUZFxFTM0Yo8cjA+eoVzqKZ6lmHBb5V55H29uLz4mP7bqe0GVX4RWnlRrzRloA/uyUNy4+J5uuBCJdZvbimk5LWVHH2ExOo0D9Y/NRQp1r4sMT6sxG+n0+vvcxlPdyAu8lSfdDAvxvvP0nvbb0ntEbkiKjxH3p/onpfwUbpZdYQ/J5PDfndN/GsoNk5wYg+eIGOZpKHDDXCpJctjjQn9tiP/g+ShLRgr01V58TYXW63jhEEBYds5idhY5QMhO2Br1+4mt6NosgDKWIDjZTZFhAwBTg0twewlRaivTtPbs1xdBVkKSVSwR5HzPmXQkhguLQJ7Js1NBfkDt6ZGBCO6dJenjEbme7iLFUljAgXNoimbhMMhVUrM50VR4rxNVRvsLrxb7Wv6e8owbmyAIfJTHx3rBic+7Hy13z92EYgWnomyRaKqn5gifPXYuLG6u/9g5eQunpHtIsLwc3L64deposLGyvXgWp7W3Tm+tHuks3QjOfrW5ttS5e6R785POxRUy6Tf4PBYHUixz8FmJ6bfsOZzrmHZY9xoUQ0OqFaaaD80qG/rkiMDYcZ3aSM0C0ZUZvalPWTOyN+sJqQaXDUOHQ+G2Tt1+KByX7gTkWpo6cBbi23ACzSnVTEHM93T5zagcxsmRwjQgmjZ7Q/FFO4rDsOjDfe6XKqXogqsyhGvYQwDY2rdH4k0meB2rW5j+mL9BYkk6d0zu/M5VG7Qbjfnk15yG34Lw+w6+Re2UColB0xQ13IPk0LRBEjJFDbss9A3iU3P4ZDLZOXmBhGN4YQhONrzEjEPrqm94ZSEmsPXzF4CQvPXuoTGydW81+PnYcByRWgCYaYzNHyKUe/JKPp8fCju6M9DQ8+M7ni/u3TcUwnAiEJbWEiDsZ4ZCsSC6I618pKJ8mX+7QDccy+X2E5ENyKXhikoFC0CtsoFqmbkJVTwo6WLt84CA0+y0+3O/JuzTTfTRhn+iiUg6DXRboMwMC2mDAWZSnR6JKgcsGrn94yu2Meq0qhtYZ4gSK3PgoO0LvU1hHPUnmS4L+xDNQFVCVFPrQz5ZL+xTXN/GcEOhIghJhB7IcfHmrxnTwDgroggymj6EatVm4ZZpjdPMkhmqN6Y8V7FMY66cajxq9QYggvzlibVmRYBXocp3JlUX5QibbXHXLtiT+otyPMQYwDbdo0qMiFVUEaoxwMNPRg2sPJY0yNzFXZm/gKJSb9ifMg2EI8gxHKjtDwc1DqbuulB5G+hXCQpfbsd6mCcfkrFye2yevFlu9xF/c540Bxab82FTPLDDlxEhY6rcZj/zAk6mfXcq7DKijpZxkLLgpFQKcyHOB5+UGstYoDBM0UqWp0lphw40+gahN9nRpD9knzPwO44cf8cp7N7figBSXIREkvrdZJVLiFFuQE8FS5ammw1RaNheUZAeL1aIUASoLIml6pEoVNlLpLN6dnPtFubnj8+QYm432bpzu7v+Ua6z8mWweOLRxung0yVSgPXu+csQbPk3V24/tqGzZrQ9HzebzSybjGJ+CTmLMkyisZqPdYeYjGl3kSKb1e6Oj3DUlSaWX+LANLlSAhRsggoCjYa/41J2cO5cgYOlwZOh8FEpWx4cqaIRZiM2WF1RreSrWPSGS7xswMWonkxPTRl6y9CUPKoHsnjPl5se5M1qpbAjjgqf7Fe5XlGjImMkG9UsEle7W2BUhlPgxcqTyHBcxSfiwmLmqXDFEWIUEFRRG+U2Uwn6MrIErlwI3wApvAHxHf0aujjIaczSeIFRIqmIy8JmCWMpaoZlxBIJ2wsBmYfXtgCU4ZfRz8blG9LPxgUaow+vGD3xi0p7PrFrNrpgVxDNLkJiUJoLKdtuoTFBgyprUKbjA1OftNM4xS/A2d45ayvLCA3cHBpZaKw7qJ3NiXjgg+KdR0n2b2pIA3IXdjLF1vOzw9AWC/3203pWQ0w0VHwKDXF7fC4aKv4+Gir+Zg3FyIqPR/ZUvhpje0+t1fin3IGh3R+GDu1wNjqJTIA5Q72QxUs7QHob5hAx/4jIxzbiZIH8EQClnrY/+dqJ+x8O7EvsW2OM22EdfruHlsDspKCgIhXNsez46yWLkyPMRHs/jqbBZizH9RQ0BkdnkSP6ZDorMh/ClDTCwwg+xhfOVqRheIsKXkUfO0XODlzhY8BQCiACJUBajuIT5AgBTTpUBY+h+QQVFH+dCgpFBSzJN6jyHv+LGECMT8nZ+dS9snhfCSNxtYK/WAo4qSlLlF4IqbTZZ6g/ufNV0oYjj60NoZVMDBheFDv6+5BoW+rdjv5GpNXUdEfkLy7DnCV0VkdVNftG3RxuxgGjVTw664n49zCyBt27K4ay4eRLA0xl/HweFXdlwffqI/tT1V2cQ9+FRgDK7DZDPSAg/uWKophqC/9mBzschf8VkBI6K+/qM/8HLroaow=="}
root = Path("/content/rmp")
for rel, b64 in BUNDLE.items():
    p = root / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(zlib.decompress(base64.b64decode(b64)))
print("Extracted", len(BUNDLE), "files to", root)


In [ ]:
# 4. 先跑 2 分钟测实测吞吐与显存（30 步；L4 24GB 若 OOM 把 --batch 16 --grad_accum 4 改为 --batch 8 --grad_accum 8）
%cd /content/rmp
!python tools/train_rmp_lm.py train --preset main_fine --out /content/drive/MyDrive/rmp_main_fine --batch 16 --T 1024 --grad_accum 4 --steps 30 --log_every 10 --save_every 30 --eval_every 30 --resume


In [ ]:
# 5. 正式预训练（默认 8 亿 token ≈ 100 CU A100；预算足可改 --tokens 2000000000；断线重跑自动 --resume 精确续训）
%cd /content/rmp
!python tools/train_rmp_lm.py train --preset main_fine --out /content/drive/MyDrive/rmp_main_fine --batch 16 --T 1024 --grad_accum 4 --tokens 800000000 --sched wsd --save_every 250 --eval_every 500 --log_every 20 --resume


In [ ]:
# 6. 训完评测：四合一诊断（竞争/分工/层级/单阶段抗遗忘）+ 多阶段真实持续学习（chat -> code）
%cd /content/rmp
!python tools/eval_rmp_138m.py /content/drive/MyDrive/rmp_main_fine/latest.pt --out /content/drive/MyDrive/rmp_main_fine/eval_4in1.json
!python tools/eval_rmp_bpe_multistage.py /content/drive/MyDrive/rmp_main_fine/latest.pt --out /content/drive/MyDrive/rmp_main_fine/eval_multistage.json
